## Install Dependencies


In [54]:
# Dependencis for the notebook
%pip install spacy pandas torch transformers huggingface_hub matplotlib ipympl rdflib sentence_transformers numpy
# optuna and optional dependencies
%pip install optuna scikit-learn plotly 
# Qwen optimization (optional) dependencies
%pip install flash-linear-attention causal-conv1d
# Mistral dependencies
%pip install accelerate


/home/rpatroni/bzk-post-processing/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
/home/rpatroni/bzk-post-processing/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
/home/rpatroni/bzk-post-processing/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.
/home/rpatroni/bzk-post-processing/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [55]:
import matplotlib.pyplot as plt
import matplotlib
import optuna.visualization
import optuna.importance
import json
from pathlib import Path
from collections import OrderedDict
import modules.llms as llm_parsers
import modules.deepparse_parser as deepparse
import modules.libpostal_client as libpostal_client
import modules.token_classifiers as token_classifiers
from modules.utils import compare_preds, format_time, natural_casing
from typing import NamedTuple
from tqdm.auto import tqdm

from typing import Callable

import abc
import contextlib
import pandas as pd
import time
import pprint
import textwrap
import numpy as np
import modules.train_ner as train_ner
import re

# Total number of addresses in the complete BZK data for calculating estimated time of processing
TOTAL_ADDRESSES = 4_394_539

REQUIRED_ENTITIES = [
    "HouseNumber",
    "StreetName",
    "City",
    "Country"
]

ALL_ENTITIES = [
    "HouseNumber",
    "StreetName",
    "Neighborhood",
    "City",
    "District",
    "State",
    "Region",
    "Country"
]

BZK_ADDRESS_FIELDS = [
    'ApplicantCurrentAddress',
    'VictimBirthPlace',
    'VictimCurrentAddress',     
    'ApplicantBirthPlace',
    'VictimDeathPlace'
]

## Set hardware accelerator


In [56]:
# List available hardware accelerators for PyTorch
import torch
available_accelerators = []
if torch.cuda.is_available():
    print("CUDA - available devices:")
    for i in range(torch.cuda.device_count()):
        print(f"  {i}: {torch.cuda.get_device_name(i)}")
        available_accelerators.append(torch.device(f'cuda:{i}'))
elif torch.accelerator.is_available(): # Support other hardware accelators
    available_accelerators.append(torch.accelerator.current_accelerator())
else:
    print("WARNING: Running on CPU")
    device = torch.device('cpu')

CUDA - available devices:
  0: NVIDIA A100 80GB PCIe


In [57]:
if available_accelerators:
    device = available_accelerators[0]
print(f"Torch version: {torch.__version__}, Device: {device}")

Torch version: 2.10.0+cu128, Device: cuda:0


## Load Dataset

Read all splits and concat them

In [58]:
csv_read_args = dict(keep_default_na=False, dtype=str, na_values=[""])

bzkopen_train = pd.read_csv("open_data/bzkopen_addresses_train.csv", **csv_read_args)
bzkopen_val = pd.read_csv("open_data/bzkopen_addresses_val.csv", **csv_read_args)
bzkopen_test = pd.read_csv("open_data/bzkopen_addresses_test.csv", **csv_read_args)
bzkopen_merged : pd.DataFrame = pd.concat(
    [bzkopen_train, bzkopen_val, bzkopen_test], 
    keys=['train', 'val', 'test'], 
    names=['split', 'row']
)
display(bzkopen_merged.sample(5))

,,field,FullAddress,UnitNumber,HouseNumber,StreetName,Neighborhood,City,District,Region,State,Country,PostalCode
split,row,,,,,,,,,,,,
train,529,VictimBirthPlace,Offenbach / Main. Hessen,NaN,NaN,NaN,NaN,Offenbach,NaN,Main,Hessen,NaN,NaN
val,129,VictimCurrentAddress,Sdeh Warburg Erif 5 b. Kfar Saba,NaN,5 b,Erif,NaN,Sdeh Warburg,NaN,Kfar Saba,NaN,NaN,NaN
train,331,ApplicantBirthPlace,Heidenheim,NaN,NaN,NaN,NaN,Heidenheim,NaN,NaN,NaN,NaN,NaN
test,69,ApplicantCurrentAddress,Peabody Mass. 112 Main Street USA,NaN,112,Main Street,NaN,Peabody,NaN,NaN,Mass.,USA,NaN
train,87,ApplicantBirthPlace,Lippstadt i.W.,NaN,NaN,NaN,NaN,Lippstadt i.W.,NaN,NaN,NaN,NaN,NaN


## Create folds



In [59]:
N_FOLDS = 10

# Set shuffle seed for reproducibility
SEED=4841762


In [60]:
# Define paths for saving models and predictions
models_path = Path("models") / "cross_val" / f"{N_FOLDS}_folds" / f"seed_{SEED}"
models_path.mkdir(parents=True, exist_ok=True)
preds_path = Path("experiments_data") / "cross_val" / f"{N_FOLDS}_folds" / f"seed_{SEED}"
preds_path.mkdir(parents=True, exist_ok=True)

In [61]:
class Fold(NamedTuple):
    fold_idx: int
    train_data: pd.DataFrame
    val_data: pd.DataFrame
    models_path: Path
    preds_path: Path

In [62]:

shuffled_bzkopen = bzkopen_merged.sample(frac=1, random_state=SEED)

fold_indices = np.array_split(shuffled_bzkopen.index, N_FOLDS)

folds = []

for fold_idx, val_indices in enumerate(fold_indices):
    train_indices = shuffled_bzkopen.index.difference(val_indices)
    assert set(train_indices).isdisjoint(set(val_indices)), "Train and validation indices overlap!"
    fold_models_path = models_path / f"fold_{fold_idx}"
    fold_models_path.mkdir(parents=True, exist_ok=True)
    fold_preds_path = preds_path / f"fold_{fold_idx}"
    fold_preds_path.mkdir(parents=True, exist_ok=True)
    folds.append(Fold(
        fold_idx, 
        shuffled_bzkopen.loc[train_indices], 
        shuffled_bzkopen.loc[val_indices],
        fold_models_path, 
        fold_preds_path
    ))
display(pd.DataFrame(
    [[len(x) for x in [fold.val_data, fold.train_data]] for fold in folds], 
    columns=['val_fold_size', 'train_fold_size'])
)

,val_fold_size,train_fold_size
0,110,981
1,109,982
2,109,982
3,109,982
4,109,982
5,109,982
6,109,982
7,109,982
8,109,982
9,109,982


# Train Spacy NER models

These models are used for the pattern example matching strategy so they will be reused over multiple different experiments. Therefore it is useful to train them in advance.

In [63]:
for fold in tqdm(folds, unit="fold"):
    output_dir = fold.models_path / 'ner_bzk'
    if output_dir.exists():
        print(f"Fold {fold.fold_idx} already has a trained model, skipping training.")
        continue
    train_ner.train(
        n_iter=30,
        output_dir=output_dir,
        train_df=fold.train_data,
        val_df=fold.val_data
    )

  0%|          | 0/10 [00:00<?, ?fold/s]

Fold 0 already has a trained model, skipping training.
Fold 1 already has a trained model, skipping training.
Fold 2 already has a trained model, skipping training.
Fold 3 already has a trained model, skipping training.
Fold 4 already has a trained model, skipping training.
Fold 5 already has a trained model, skipping training.
Fold 6 already has a trained model, skipping training.
Fold 7 already has a trained model, skipping training.
Fold 8 already has a trained model, skipping training.
Fold 9 already has a trained model, skipping training.


In [64]:
# Metrics on the required entities (Country, City, StreetName, HouseNumber)
required_results = OrderedDict()
specific_results = OrderedDict()
predictions_dfs = OrderedDict()

class FoldEvaluator:
    def __init__(self, model_name : str, supported_entities : list[str]):
        # Metrics on the required entities (Country, City, StreetName, HouseNumber)
        self._required_metric_records = None
        self._specific_metric_records = None
        self._preds_per_fold = None
        self.required_metrics = None
        self.specific_metrics = None
        self.model_name = model_name
        self.supported_entities = supported_entities
    
    def __enter__(self):
        self.required_metrics = None
        self.specific_metrics = None
        self._required_metric_records = []
        self._specific_metric_records = []
        self._preds_per_fold = []
        return self
    
    def __exit__(self, exc_type, exc_value, traceback):
        global required_results, specific_results
        self.required_metrics = pd.DataFrame(self._required_metric_records).aggregate(['mean', 'std']).unstack()
        self.specific_metrics = pd.DataFrame(self._specific_metric_records).aggregate(['mean', 'std']).unstack()
        self.preds_per_fold = pd.concat(self._preds_per_fold, ignore_index=False, sort=True)
        required_results[self.model_name] = self.required_metrics
        specific_results[self.model_name] = self.specific_metrics
        predictions_dfs[self.model_name] = self.preds_per_fold
        print(f"{self.model_name} - Metrics for Country City Street House:")
        display(self.required_metrics)
        print(f"{self.model_name} - Specific Metrics:")
        display(self.specific_metrics.unstack(level=0))
        self._required_metric_records = None
        self._specific_metric_records = None
        self._preds_per_fold = None
    def _record_predictions(self, fold : Fold, preds : list[dict]):
        preds_df = pd.DataFrame(preds)
        fold_metrics = compare_preds(preds_df, fold.val_data, target_columns=REQUIRED_ENTITIES)
        self._required_metric_records.append(pd.Series(fold_metrics))
        specific_fold_metrics = OrderedDict()
        for entity in self.supported_entities:
            entity_metrics = compare_preds(preds_df, fold.val_data, target_columns=[entity])
            specific_fold_metrics[entity] = pd.Series(entity_metrics)
        for bzk_field in BZK_ADDRESS_FIELDS:
            mask = fold.val_data['field'] == bzk_field
            field_metrics = compare_preds(preds_df[mask.reset_index(drop=True)], fold.val_data[mask], target_columns=REQUIRED_ENTITIES)
            specific_fold_metrics[bzk_field] = pd.Series(field_metrics)
        self._specific_metric_records.append(pd.concat(specific_fold_metrics, names=['Field/Entity', 'Metric']))
        preds_df.index = fold.val_data.index
        self._preds_per_fold.append(preds_df)

    def load_preds_and_skip(self, folds : list[Fold]) -> list[Fold]:
        """
        For each fold, load its predictions if available on disk.
        Returns the folds for which predictions were not loaded
        """
        unloaded_folds = []
        for fold in folds:
            preds_file = fold.preds_path / self.model_name / "preds.json"
            if preds_file.exists():
                print(f"Fold {fold.fold_idx} already has predictions for {self.model_name}, loading from file.")
                with open(preds_file, "r") as f:
                    preds = json.load(f)
                self._record_predictions(fold, preds)
            else:
                unloaded_folds.append(fold)
        return unloaded_folds
            
    
    def run_on_fold(self, fold : Fold, model):
        assert self._required_metric_records is not None and self._specific_metric_records is not None, "FoldMetricRecorder must be used as a context manager"
        preds_file = fold.preds_path / self.model_name / "preds.json"
        preds_file.parent.mkdir(parents=True, exist_ok=True)
        preds = model.parse_addresses(fold.val_data['FullAddress'].tolist())
        with open(preds_file, "w") as f:
            json.dump(preds, f, indent=4)
        self._record_predictions(fold, preds)
    

In [65]:
with FoldEvaluator("libpostal", libpostal_client.LIBPOSTAL_LABEL_MAPPING.values()) as evaluator:
    folds_to_process = evaluator.load_preds_and_skip(folds)
    if folds_to_process:
        model = libpostal_client.LibpostalClient()
        for fold in tqdm(folds_to_process, unit="fold"):
            evaluator.run_on_fold(fold, model)
        model.close()

Fold 0 already has predictions for libpostal, loading from file.
Fold 1 already has predictions for libpostal, loading from file.
Fold 2 already has predictions for libpostal, loading from file.
Fold 3 already has predictions for libpostal, loading from file.
Fold 4 already has predictions for libpostal, loading from file.
Fold 5 already has predictions for libpostal, loading from file.
Fold 6 already has predictions for libpostal, loading from file.
Fold 7 already has predictions for libpostal, loading from file.
Fold 8 already has predictions for libpostal, loading from file.
Fold 9 already has predictions for libpostal, loading from file.
libpostal - Metrics for Country City Street House:


accuracy                   mean    0.690419
                           std     0.017138
precision                  mean    0.643551
                           std     0.041922
recall                     mean    0.420811
                           std     0.033534
f1                         mean    0.508282
                           std     0.033117
accuracy_with_tol_1        mean    0.700507
                           std     0.017059
accuracy_with_tol_2        mean    0.709664
                           std     0.016274
accuracy_with_tol_3        mean    0.725471
                           std     0.015642
accuracy_with_tol_4        mean    0.744033
                           std     0.014143
average_levenshtein        mean    2.486159
                           std     0.175770
average_similarity         mean    0.731492
                           std     0.016348
average_levenshtein_match  mean    3.185408
                           std     0.266792
average_similarity_match   mean 

libpostal - Specific Metrics:


Field/Entity                    HouseNumber  StreetName       City     State  \
Metric                                                                         
accuracy                  mean     0.872577    0.766289   0.328107  0.944078   
                          std      0.023948    0.021029   0.036270  0.024656   
precision                 mean     0.750388    0.512468   0.643894  0.647540   
                          std      0.055830    0.048002   0.077357  0.327424   
recall                    mean     0.684340    0.499998   0.307167  0.412168   
                          std      0.064354    0.053223   0.037001  0.224561   
f1                        mean     0.715446    0.505154   0.414210  0.498642   
                          std      0.058443    0.044456   0.040771  0.260635   
accuracy_with_tol_1       mean     0.880834    0.768123   0.348274  0.948657   
                          std      0.025990    0.023044   0.039291  0.024941   
accuracy_with_tol_2       mean     0.907398    0.769033   0.350108  0.949575   
                          std      0.027574    0.022215   0.038052  0.024959   
accuracy_with_tol_3       mean     0.923895    0.775446   0.362944  0.952327   
                          std      0.027421    0.023657   0.037439  0.026954   
accuracy_with_tol_4       mean     0.951393    0.784612   0.385855  0.973403   
                          std      0.023731    0.024053   0.038171  0.025019   
average_levenshtein       mean     0.570325    2.864295   5.430309  0.318190   
                          std      0.167511    0.444221   0.446035  0.191310   
average_similarity        mean     0.897927    0.836936   0.386810  0.948072   
                          std      0.018176    0.019830   0.043088  0.023922   
average_levenshtein_match mean     0.599688    3.137459  12.171985  0.339506   
                          std      0.176455    0.467631   2.139289  0.211951   
average_similarity_match  mean     0.943807    0.917137   0.853119  0.998406   
                          std      0.017647    0.019846   0.050217  0.001963   
no_match_rate             mean     0.048582    0.087081   0.545463  0.050425   
                          std      0.010650    0.028456   0.056391  0.023412   

Field/Entity                     Country  PostalCode  ApplicantCurrentAddress  \
Metric                                                                          
accuracy                  mean  0.794704    0.755963                 0.527806   
                          std   0.025451    0.398780                 0.032185   
precision                 mean  0.752356    0.050000                 0.642446   
                          std   0.128574    0.158114                 0.059837   
recall                    mean  0.321969    0.020000                 0.411544   
                          std   0.079218    0.063246                 0.046130   
f1                        mean  0.447938    0.028571                 0.500826   
                          std   0.092844    0.090351                 0.047638   
accuracy_with_tol_1       mean  0.804796    0.755963                 0.537128   
                          std   0.023780    0.398780                 0.026840   
accuracy_with_tol_2       mean  0.812118    0.760550                 0.551349   
                          std   0.023981    0.401217                 0.031533   
accuracy_with_tol_3       mean  0.839600    0.760550                 0.576476   
                          std   0.026014    0.401217                 0.032657   
accuracy_with_tol_4       mean  0.854270    0.765138                 0.611129   
                          std   0.024576    0.403535                 0.026344   
average_levenshtein       mean  1.079708    0.537623                 3.980700   
                          std   0.155666    0.261915                 0.415936   
average_similarity        mean  0.804297    0.756575                 0.595207   
                          std   0.024364    0.399044      

In [66]:
with FoldEvaluator("deepparse", deepparse.DEEPPARSE_LABEL_MAPPING.values()) as evaluator:
    folds_to_process = evaluator.load_preds_and_skip(folds)
    if folds_to_process:
        model = deepparse.DeepParseParser(device=device)
        for fold in tqdm(folds_to_process, unit="fold"):
            evaluator.run_on_fold(fold, model)
        del model

Fold 0 already has predictions for deepparse, loading from file.
Fold 1 already has predictions for deepparse, loading from file.
Fold 2 already has predictions for deepparse, loading from file.
Fold 3 already has predictions for deepparse, loading from file.
Fold 4 already has predictions for deepparse, loading from file.
Fold 5 already has predictions for deepparse, loading from file.
Fold 6 already has predictions for deepparse, loading from file.
Fold 7 already has predictions for deepparse, loading from file.
Fold 8 already has predictions for deepparse, loading from file.
Fold 9 already has predictions for deepparse, loading from file.
deepparse - Metrics for Country City Street House:


accuracy                   mean    0.578847
                           std     0.025272
precision                  mean    0.335184
                           std     0.032355
recall                     mean    0.281208
                           std     0.028222
f1                         mean    0.305686
                           std     0.029372
accuracy_with_tol_1        mean    0.585496
                           std     0.026167
accuracy_with_tol_2        mean    0.600617
                           std     0.025068
accuracy_with_tol_3        mean    0.616651
                           std     0.020833
accuracy_with_tol_4        mean    0.638188
                           std     0.021347
average_levenshtein        mean    3.891282
                           std     0.229668
average_similarity         mean    0.655167
                           std     0.024832
average_levenshtein_match  mean    5.029349
                           std     0.385789
average_similarity_match   mean 

deepparse - Specific Metrics:


Field/Entity                    HouseNumber  StreetName       City   Country  \
Metric                                                                         
accuracy                  mean     0.870751    0.482143   0.265847  0.696647   
                          std      0.029834    0.044953   0.038179  0.035875   
precision                 mean     0.842562    0.050316   0.333524  0.273492   
                          std      0.060412    0.028491   0.055806  0.160873   
recall                    mean     0.674083    0.066804   0.268224  0.074158   
                          std      0.062977    0.036471   0.045888  0.040757   
f1                        mean     0.747589    0.057268   0.297124  0.115678   
                          std      0.053918    0.031776   0.049732  0.064536   
accuracy_with_tol_1       mean     0.888173    0.482143   0.274103  0.697565   
                          std      0.026245    0.044953   0.038339  0.034687   
accuracy_with_tol_2       mean     0.941326    0.483061   0.279600  0.698482   
                          std      0.026044    0.046365   0.039481  0.034798   
accuracy_with_tol_3       mean     0.961493    0.485813   0.286931  0.732369   
                          std      0.014877    0.045364   0.041512  0.037286   
accuracy_with_tol_4       mean     0.977073    0.497715   0.307998  0.769967   
                          std      0.011656    0.050677   0.038188  0.027165   
average_levenshtein       mean     0.443753    6.666489   6.650175  1.804712   
                          std      0.131512    0.519804   0.527152  0.200529   
average_similarity        mean     0.880711    0.629824   0.408921  0.701211   
                          std      0.030532    0.036870   0.041588  0.035896   
average_levenshtein_match mean     0.497240    8.056275  10.018333  2.576624   
                          std      0.154090    0.802308   1.107247  0.390839   
average_similarity_match  mean     0.980545    0.758960   0.613165  0.994861   
                          std      0.009139    0.024823   0.046116  0.004749   
no_match_rate             mean     0.101743    0.170467   0.333586  0.295096   
                          std      0.032209    0.032003   0.036103  0.037417   

Field/Entity                    PostalCode  ApplicantCurrentAddress  \
Metric                                                                
accuracy                  mean    0.821201                 0.360272   
                          std     0.289926                 0.024666   
precision                 mean    0.020000                 0.322538   
                          std     0.063246                 0.037912   
recall                    mean    0.020000                 0.238151   
                          std     0.063246                 0.029357   
f1                        mean    0.020000                 0.273866   
                          std     0.063246                 0.032584   
accuracy_with_tol_1       mean    0.826706                 0.369518   
                          std     0.292052                 0.026195   
accuracy_with_tol_2       mean    0.837698                 0.392935   
                          std     0.295539                 0.022508   
accuracy_with_tol_3       mean    0.844120                 0.418829   
                          std     0.297616                 0.026651   
accuracy_with_tol_4       mean    0.853286                 0.452301   
                          std     0.300678                 0.027389   
average_levenshtein       mean    0.581626                 6.408708   
                          std     0.309334                 0.368108   
average_similarity        mean    0.821444                 0.475258   
                          std     0.289982                 0.022444   
average_levenshtein_match mean    0.621334                 9.407359   
                          std     0.390585                 0.523264   
average_similarity_match  mean    0.897219             

In [67]:
with FoldEvaluator("xlm-roberta-large", token_classifiers.FIGHTING_CRIME_LABEL_MAPPING.values()) as evaluator:
    folds_to_process = evaluator.load_preds_and_skip(folds)
    if folds_to_process:
        model = token_classifiers.TokenClassifierAddressParser(
            model_name="hm-haitham/xlm-roberta-large-address-parser", 
            device=device,
            aggregation_strategy="average"
        )
        for fold in tqdm(folds_to_process, unit="fold"):
            evaluator.run_on_fold(fold, model)
        del model

Fold 0 already has predictions for xlm-roberta-large, loading from file.
Fold 1 already has predictions for xlm-roberta-large, loading from file.
Fold 2 already has predictions for xlm-roberta-large, loading from file.
Fold 3 already has predictions for xlm-roberta-large, loading from file.
Fold 4 already has predictions for xlm-roberta-large, loading from file.
Fold 5 already has predictions for xlm-roberta-large, loading from file.
Fold 6 already has predictions for xlm-roberta-large, loading from file.
Fold 7 already has predictions for xlm-roberta-large, loading from file.
Fold 8 already has predictions for xlm-roberta-large, loading from file.
Fold 9 already has predictions for xlm-roberta-large, loading from file.
xlm-roberta-large - Metrics for Country City Street House:


accuracy                   mean    0.803415
                           std     0.018579
precision                  mean    0.713449
                           std     0.028597
recall                     mean    0.657483
                           std     0.020281
f1                         mean    0.684189
                           std     0.022231
accuracy_with_tol_1        mean    0.814867
                           std     0.018525
accuracy_with_tol_2        mean    0.822425
                           std     0.017458
accuracy_with_tol_3        mean    0.834112
                           std     0.019003
accuracy_with_tol_4        mean    0.850152
                           std     0.016086
average_levenshtein        mean    1.458849
                           std     0.193665
average_similarity         mean    0.837693
                           std     0.016631
average_levenshtein_match  mean    1.635630
                           std     0.232367
average_similarity_match   mean 

xlm-roberta-large - Specific Metrics:


Field/Entity                     Country     State      City  StreetName  \
Metric                                                                     
accuracy                  mean  0.903778  0.926706  0.526172    0.872611   
                          std   0.021607  0.024300  0.041898    0.031826   
precision                 mean  0.784409  0.347976  0.615296    0.751668   
                          std   0.057609  0.185643  0.048869    0.055668   
recall                    mean  0.805284  0.255523  0.530120    0.735048   
                          std   0.041566  0.148836  0.041382    0.056194   
f1                        mean  0.793565  0.291726  0.569159    0.742718   
                          std   0.038570  0.156687  0.042440    0.051931   
accuracy_with_tol_1       mean  0.918415  0.934037  0.527089    0.885446   
                          std   0.018611  0.021807  0.043237    0.031162   
accuracy_with_tol_2       mean  0.922994  0.935872  0.528007    0.886364   
                          std   0.018995  0.023925  0.044725    0.030536   
accuracy_with_tol_3       mean  0.933995  0.945947  0.543595    0.893695   
                          std   0.025168  0.022555  0.049346    0.028980   
accuracy_with_tol_4       mean  0.952335  0.964262  0.562844    0.899191   
                          std   0.018252  0.016972  0.043574    0.033155   
average_levenshtein       mean  0.455605  0.397723  3.964629    1.129158   
                          std   0.146963  0.168705  0.500166    0.369874   
average_similarity        mean  0.911383  0.934297  0.604168    0.904710   
                          std   0.020983  0.022510  0.041532    0.025849   
average_levenshtein_match mean  0.500419  0.424209  5.178573    1.218869   
                          std   0.165941  0.185094  0.781968    0.420779   
average_similarity_match  mean  0.996303  0.989566  0.785324    0.970436   
                          std   0.002663  0.005312  0.033222    0.009997   
no_match_rate             mean  0.085238  0.055880  0.230984    0.067815   
                          std   0.020737  0.020786  0.034563    0.020774   

Field/Entity                    HouseNumber  ApplicantCurrentAddress  \
Metric                                                                 
accuracy                  mean     0.911101                 0.701201   
                          std      0.018814                 0.025970   
precision                 mean     0.836146                 0.720795   
                          std      0.038600                 0.034148   
recall                    mean     0.779569                 0.648900   
                          std      0.042833                 0.029816   
f1                        mean     0.806542                 0.682608   
                          std      0.037317                 0.027168   
accuracy_with_tol_1       mean     0.928515                 0.721599   
                          std      0.017676                 0.028938   
accuracy_with_tol_2       mean     0.952335                 0.731733   
                          std      0.015479                 0.024820   
accuracy_with_tol_3       mean     0.965163                 0.750545   
                          std      0.012846                 0.027665   
accuracy_with_tol_4       mean     0.986239                 0.779273   
                          std      0.009909                 0.023705   
average_levenshtein       mean     0.286005                 2.267925   
                          std      0.079351                 0.337558   
average_similarity        mean     0.930511                 0.752413   
                          std      0.015736                 0.023257   
average_levenshtein_match mean     0.299551                 2.674984   
                          std      0.084733                 0.438667   
average_similarity_match  mean     0.972439                 0.885567   
                          std      0.009039                 0.018022   

In [68]:
with FoldEvaluator("SpaCy", supported_entities=ALL_ENTITIES) as evaluator:
    folds_to_process = evaluator.load_preds_and_skip(folds)
    if folds_to_process:
        for fold in tqdm(folds_to_process, unit="fold"):
            model = train_ner.SpaCyAddressParser(model_dir=fold.models_path / 'ner_bzk')
            evaluator.run_on_fold(fold, model)

Fold 0 already has predictions for SpaCy, loading from file.
Fold 1 already has predictions for SpaCy, loading from file.
Fold 2 already has predictions for SpaCy, loading from file.
Fold 3 already has predictions for SpaCy, loading from file.
Fold 4 already has predictions for SpaCy, loading from file.
Fold 5 already has predictions for SpaCy, loading from file.
Fold 6 already has predictions for SpaCy, loading from file.
Fold 7 already has predictions for SpaCy, loading from file.
Fold 8 already has predictions for SpaCy, loading from file.
Fold 9 already has predictions for SpaCy, loading from file.
SpaCy - Metrics for Country City Street House:


accuracy                   mean    0.870334
                           std     0.028139
precision                  mean    0.822052
                           std     0.036197
recall                     mean    0.778655
                           std     0.040605
f1                         mean    0.799575
                           std     0.036357
accuracy_with_tol_1        mean    0.880640
                           std     0.025881
accuracy_with_tol_2        mean    0.888887
                           std     0.024300
accuracy_with_tol_3        mean    0.896445
                           std     0.019492
accuracy_with_tol_4        mean    0.904229
                           std     0.019795
average_levenshtein        mean    1.058955
                           std     0.235877
average_similarity         mean    0.899923
                           std     0.024019
average_levenshtein_match  mean    1.140486
                           std     0.274580
average_similarity_match   mean 

SpaCy - Specific Metrics:


Field/Entity                    HouseNumber  StreetName  Neighborhood  \
Metric                                                                  
accuracy                  mean     0.891860    0.847857      0.892761   
                          std      0.034470    0.030268      0.027016   
precision                 mean     0.789920    0.744952      0.264881   
                          std      0.053274    0.047841      0.186323   
recall                    mean     0.742714    0.656649      0.184488   
                          std      0.081686    0.067641      0.214997   
f1                        mean     0.764610    0.697201      0.187857   
                          std      0.064721    0.054998      0.163503   
accuracy_with_tol_1       mean     0.912027    0.850609      0.892761   
                          std      0.029314    0.029263      0.027016   
accuracy_with_tol_2       mean     0.941359    0.852444      0.893678   
                          std      0.022048    0.029380      0.028082   
accuracy_with_tol_3       mean     0.953278    0.859766      0.894595   
                          std      0.013840    0.027338      0.030026   
accuracy_with_tol_4       mean     0.961510    0.870767      0.897339   
                          std      0.008391    0.030684      0.030832   
average_levenshtein       mean     0.461068    1.679049      1.075096   
                          std      0.129682    0.432650      0.286506   
average_similarity        mean     0.918580    0.882836      0.894251   
                          std      0.028796    0.027296      0.026285   
average_levenshtein_match mean     0.485154    1.850093      1.209231   
                          std      0.143924    0.519428      0.346492   
average_similarity_match  mean     0.961798    0.966000      0.998584   
                          std      0.008685    0.005936      0.002293   
no_match_rate             mean     0.044921    0.086155      0.104487   
                          std      0.029167    0.025273      0.025999   

Field/Entity                        City  District     State    Region  \
Metric                                                                   
accuracy                  mean  0.795663  0.909266  0.948657  0.000000   
                          std   0.055886  0.019022  0.019466  0.000000   
precision                 mean  0.835991  0.547628  0.721984  0.000000   
                          std   0.052148  0.077316  0.240292  0.000000   
recall                    mean  0.821513  0.596771  0.457013  0.000000   
                          std   0.037144  0.183477  0.189180  0.000000   
f1                        mean  0.828514  0.551756  0.546344  0.000000   
                          std   0.043345  0.102459  0.195257  0.000000   
accuracy_with_tol_1       mean  0.808482  0.910183  0.948657  0.000000   
                          std   0.050060  0.019188  0.019466  0.000000   
accuracy_with_tol_2       mean  0.809391  0.913845  0.951410  0.000000   
                          std   0.049043  0.017380  0.017883  0.000000   
accuracy_with_tol_3       mean  0.814896  0.914762  0.954162  0.000000   
                          std   0.046268  0.016744  0.018367  0.000000   
accuracy_with_tol_4       mean  0.822219  0.923011  0.975238  0.000000   
                          std   0.044362  0.016271  0.013721  0.000000   
average_levenshtein       mean  1.756606  0.712977  0.303503  0.657239   
                          std   0.485198  0.208413  0.134058  0.104431   
average_similarity        mean  0.846007  0.913475  0.950521  0.000000   
                          std   0.048921  0.016818  0.019823  0.000000   
average_levenshtein_match mean  1.966472  0.780807  0.320899  0.000000   
                          std   0.629034  0.238090  0.147622  0.000000   
average_similarity_match  mean  0.934978  0.996583  0.997189  0.000000   
                          std   0.016039  0.004288  0.005546  0.000000   
no_match_rate             mean 

In [69]:
prompt0_template = llm_parsers.JsonDictPromptTemplate(Path("prompts/prompt_0.txt").read_text())
supported_entities = [
    "HouseNumber",
    "StreetName",
    "Neighborhood",
    "City",
    "State",
    "District",
    "Country"
]

with FoldEvaluator("Llama-3-8B-prompt0-0shot", supported_entities) as evaluator:
    folds_to_process = evaluator.load_preds_and_skip(folds)
    if folds_to_process:
        model = llm_parsers.LlamaAddressParsingModel(
            model_name="meta-llama/Meta-Llama-3-8B-Instruct", 
            device=device,
            example_strategy=llm_parsers.ZeroShot(),
            prompt=prompt0_template
        )
        for fold in tqdm(folds_to_process, unit="fold"):
            evaluator.run_on_fold(fold, model)
        del model

Fold 0 already has predictions for Llama-3-8B-prompt0-0shot, loading from file.
Fold 1 already has predictions for Llama-3-8B-prompt0-0shot, loading from file.
Fold 2 already has predictions for Llama-3-8B-prompt0-0shot, loading from file.
Fold 3 already has predictions for Llama-3-8B-prompt0-0shot, loading from file.
Fold 4 already has predictions for Llama-3-8B-prompt0-0shot, loading from file.
Fold 5 already has predictions for Llama-3-8B-prompt0-0shot, loading from file.
Fold 6 already has predictions for Llama-3-8B-prompt0-0shot, loading from file.
Fold 7 already has predictions for Llama-3-8B-prompt0-0shot, loading from file.
Fold 8 already has predictions for Llama-3-8B-prompt0-0shot, loading from file.
Fold 9 already has predictions for Llama-3-8B-prompt0-0shot, loading from file.
Llama-3-8B-prompt0-0shot - Metrics for Country City Street House:


accuracy                   mean    0.825851
                           std     0.020988
precision                  mean    0.744172
                           std     0.030934
recall                     mean    0.791623
                           std     0.031936
f1                         mean    0.767079
                           std     0.030214
accuracy_with_tol_1        mean    0.836387
                           std     0.017049
accuracy_with_tol_2        mean    0.842575
                           std     0.017267
accuracy_with_tol_3        mean    0.853580
                           std     0.018499
accuracy_with_tol_4        mean    0.868236
                           std     0.019436
average_levenshtein        mean    1.346768
                           std     0.198673
average_similarity         mean    0.852295
                           std     0.016042
average_levenshtein_match  mean    1.527551
                           std     0.238051
average_similarity_match   mean 

Llama-3-8B-prompt0-0shot - Specific Metrics:


Field/Entity                    HouseNumber  StreetName  Neighborhood  \
Metric                                                                  
accuracy                  mean     0.928499    0.768115      0.863436   
                          std      0.013575    0.039793      0.019960   
precision                 mean     0.830123    0.576350      0.263822   
                          std      0.028499    0.060303      0.107727   
recall                    mean     0.891240    0.792447      0.365974   
                          std      0.040384    0.068313      0.116181   
f1                        mean     0.859413    0.666986      0.297360   
                          std      0.032086    0.062739      0.108956   
accuracy_with_tol_1       mean     0.931251    0.792852      0.863436   
                          std      0.011672    0.026355      0.019960   
accuracy_with_tol_2       mean     0.945913    0.797440      0.867098   
                          std      0.012603    0.022589      0.023763   
accuracy_with_tol_3       mean     0.956914    0.811201      0.869842   
                          std      0.015029    0.022769      0.023223   
accuracy_with_tol_4       mean     0.970651    0.827681      0.875338   
                          std      0.015485    0.023994      0.022587   
average_levenshtein       mean     0.364871    1.838540      1.290475   
                          std      0.101620    0.298768      0.240063   
average_similarity        mean     0.938137    0.814848      0.869218   
                          std      0.012084    0.025081      0.020506   
average_levenshtein_match mean     0.382347    2.193867      1.475895   
                          std      0.109524    0.405130      0.303443   
average_similarity_match  mean     0.981323    0.968449      0.989968   
                          std      0.007486    0.011960      0.006396   
no_match_rate             mean     0.043995    0.158557      0.121902   
                          std      0.011271    0.025502      0.023277   

Field/Entity                        City     State  District   Country  \
Metric                                                                   
accuracy                  mean  0.735071  0.906530  0.876264  0.871718   
                          std   0.050687  0.030111  0.037000  0.028736   
precision                 mean  0.823828  0.437618  0.081905  0.705271   
                          std   0.037669  0.097482  0.135671  0.087532   
recall                    mean  0.746225  0.822024  0.030833  0.805413   
                          std   0.052523  0.091619  0.050621  0.064762   
f1                        mean  0.782733  0.565653  0.044541  0.750663   
                          std   0.043548  0.091017  0.072966  0.073241   
accuracy_with_tol_1       mean  0.743311  0.908365  0.877181  0.878132   
                          std   0.046190  0.027912  0.037982  0.026763   
accuracy_with_tol_2       mean  0.744229  0.911118  0.880842  0.882719   
                          std   0.046879  0.029544  0.038197  0.025935   
accuracy_with_tol_3       mean  0.749733  0.918424  0.883586  0.896472   
                          std   0.050799  0.027842  0.039669  0.026371   
accuracy_with_tol_4       mean  0.757064  0.935847  0.893678  0.917548   
                          std   0.049342  0.018313  0.034911  0.024606   
average_levenshtein       mean  2.100075  0.776439  0.955913  1.083586   
                          std   0.460401  0.319447  0.342131  0.479509   
average_similarity        mean  0.772510  0.908210  0.880757  0.883684   
                          std   0.043887  0.029847  0.036122  0.024295   
average_levenshtein_match mean  2.514679  0.861210  1.089698  1.221439   
                          std   0.646328  0.373393  0.436819  0.555452   
average_similarity_match  mean  0.917984  0.996879  0.991483  0.990755   
                          std   0.027568  0.005091  0.007434  0.007292   
no_match_rate             mean 

In [70]:
supported_entities = [
    "HouseNumber",
    "StreetName",
    "Neighborhood",
    "City",
    "State",
    "District",
    "Country"
]

with FoldEvaluator("Mistral-3-8B-Instruct-prompt0-0shot", supported_entities) as evaluator:
    folds_to_process = evaluator.load_preds_and_skip(folds)
    if folds_to_process:
        model = llm_parsers.MistralAddressParsingModel(
            model_name="mistralai/Ministral-3-8B-Instruct-2512", 
            device=device,
            example_strategy=llm_parsers.ZeroShot(),
            prompt=prompt0_template
        )
        for fold in tqdm(folds_to_process, unit="fold"):
            evaluator.run_on_fold(fold, model)
        del model

Fold 0 already has predictions for Mistral-3-8B-Instruct-prompt0-0shot, loading from file.
Fold 1 already has predictions for Mistral-3-8B-Instruct-prompt0-0shot, loading from file.
Fold 2 already has predictions for Mistral-3-8B-Instruct-prompt0-0shot, loading from file.
Fold 3 already has predictions for Mistral-3-8B-Instruct-prompt0-0shot, loading from file.
Fold 4 already has predictions for Mistral-3-8B-Instruct-prompt0-0shot, loading from file.
Fold 5 already has predictions for Mistral-3-8B-Instruct-prompt0-0shot, loading from file.
Fold 6 already has predictions for Mistral-3-8B-Instruct-prompt0-0shot, loading from file.
Fold 7 already has predictions for Mistral-3-8B-Instruct-prompt0-0shot, loading from file.
Fold 8 already has predictions for Mistral-3-8B-Instruct-prompt0-0shot, loading from file.
Fold 9 already has predictions for Mistral-3-8B-Instruct-prompt0-0shot, loading from file.
Mistral-3-8B-Instruct-prompt0-0shot - Metrics for Country City Street House:


accuracy                   mean    0.827448
                           std     0.012935
precision                  mean    0.773940
                           std     0.025574
recall                     mean    0.689833
                           std     0.027581
f1                         mean    0.729327
                           std     0.024598
accuracy_with_tol_1        mean    0.836153
                           std     0.010200
accuracy_with_tol_2        mean    0.843255
                           std     0.011911
accuracy_with_tol_3        mean    0.881297
                           std     0.009951
accuracy_with_tol_4        mean    0.894581
                           std     0.010534
average_levenshtein        mean    1.120811
                           std     0.088132
average_similarity         mean    0.875589
                           std     0.009463
average_levenshtein_match  mean    1.239139
                           std     0.112728
average_similarity_match   mean 

Mistral-3-8B-Instruct-prompt0-0shot - Specific Metrics:


Field/Entity                    HouseNumber  StreetName  Neighborhood  \
Metric                                                                  
accuracy                  mean     0.945029    0.794679      0.916564   
                          std      0.016614    0.023006      0.028234   
precision                 mean     0.880915    0.526847      0.489127   
                          std      0.031562    0.049666      0.308572   
recall                    mean     0.871592    0.549540      0.405296   
                          std      0.040024    0.051071      0.190764   
f1                        mean     0.875991    0.537773      0.433558   
                          std      0.032890    0.049297      0.228618   
accuracy_with_tol_1       mean     0.947781    0.802010      0.917481   
                          std      0.016056    0.022610      0.029385   
accuracy_with_tol_2       mean     0.961510    0.807515      0.919316   
                          std      0.013514    0.022479      0.028673   
accuracy_with_tol_3       mean     0.974337    0.924854      0.920234   
                          std      0.010411    0.019648      0.027077   
accuracy_with_tol_4       mean     0.981660    0.934929      0.922060   
                          std      0.009679    0.017507      0.029463   
average_levenshtein       mean     0.251093    1.182285      0.778432   
                          std      0.089494    0.288797      0.278011   
average_similarity        mean     0.956359    0.909561      0.920441   
                          std      0.014595    0.016927      0.026070   
average_levenshtein_match mean     0.257517    1.242902      0.844890   
                          std      0.093246    0.324179      0.314383   
average_similarity_match  mean     0.977833    0.952352      0.992224   
                          std      0.005966    0.006905      0.009640   
no_match_rate             mean     0.021985    0.044904      0.072427   
                          std      0.011558    0.018036      0.020990   

Field/Entity                        City     State  District   Country  \
Metric                                                                   
accuracy                  mean  0.649842  0.963344  0.862494  0.920242   
                          std   0.050910  0.011406  0.031259  0.029700   
precision                 mean  0.842443  0.731221  0.262152  0.810283   
                          std   0.039282  0.126040  0.126186  0.067179   
recall                    mean  0.635019  0.706898  0.316205  0.816775   
                          std   0.056610  0.135930  0.140028  0.069901   
f1                        mean  0.723606  0.708391  0.286001  0.813060   
                          std   0.049061  0.091119  0.132267  0.065362   
accuracy_with_tol_1       mean  0.670000  0.964262  0.864320  0.924821   
                          std   0.043235  0.010051  0.032396  0.026279   
accuracy_with_tol_2       mean  0.677339  0.964262  0.866155  0.926656   
                          std   0.037827  0.010051  0.031924  0.026346   
accuracy_with_tol_3       mean  0.686514  0.967923  0.869825  0.939483   
                          std   0.039685  0.012409  0.028375  0.021317   
accuracy_with_tol_4       mean  0.709425  0.972502  0.879917  0.952310   
                          std   0.039293  0.012233  0.028549  0.023632   
average_levenshtein       mean  2.576714  0.285888  1.254204  0.473153   
                          std   0.360895  0.126218  0.306850  0.198244   
average_similarity        mean  0.703644  0.966898  0.875495  0.932793   
                          std   0.039087  0.009700  0.028160  0.027193   
average_levenshtein_match mean  3.487133  0.294705  1.416992  0.505738   
                          std   0.630887  0.131374  0.379807  0.224374   
average_similarity_match  mean  0.945239  0.994261  0.983603  0.988032   
                          std   0.017964  0.007279  0.009054  0.005721   
no_match_rate             mean 

In [71]:
supported_entities = [
    "HouseNumber",
    "StreetName",
    "Neighborhood",
    "City",
    "State",
    "District",
    "Country"
]

with FoldEvaluator("DeepSeek-R1-Llama-8B-prompt0-0shot", supported_entities) as evaluator:
    folds_to_process = evaluator.load_preds_and_skip(folds)
    if folds_to_process:
        model = llm_parsers.DeepSeekAddressParsingModel(
            model_name="deepseek-ai/DeepSeek-R1-Distill-Llama-8B", 
            device=device,
            example_strategy=llm_parsers.ZeroShot(),
            prompt=prompt0_template
        )
        for fold in tqdm(folds_to_process, unit="fold"):
            evaluator.run_on_fold(fold, model)
        del model

Fold 0 already has predictions for DeepSeek-R1-Llama-8B-prompt0-0shot, loading from file.
Fold 1 already has predictions for DeepSeek-R1-Llama-8B-prompt0-0shot, loading from file.
Fold 2 already has predictions for DeepSeek-R1-Llama-8B-prompt0-0shot, loading from file.
Fold 3 already has predictions for DeepSeek-R1-Llama-8B-prompt0-0shot, loading from file.
Fold 4 already has predictions for DeepSeek-R1-Llama-8B-prompt0-0shot, loading from file.
Fold 5 already has predictions for DeepSeek-R1-Llama-8B-prompt0-0shot, loading from file.
Fold 6 already has predictions for DeepSeek-R1-Llama-8B-prompt0-0shot, loading from file.
Fold 7 already has predictions for DeepSeek-R1-Llama-8B-prompt0-0shot, loading from file.
Fold 8 already has predictions for DeepSeek-R1-Llama-8B-prompt0-0shot, loading from file.
Fold 9 already has predictions for DeepSeek-R1-Llama-8B-prompt0-0shot, loading from file.
DeepSeek-R1-Llama-8B-prompt0-0shot - Metrics for Country City Street House:


accuracy                   mean    0.000000
                           std     0.000000
precision                  mean    0.000000
                           std     0.000000
recall                     mean    0.000000
                           std     0.000000
f1                         mean    0.000000
                           std     0.000000
accuracy_with_tol_1        mean    0.000000
                           std     0.000000
accuracy_with_tol_2        mean    0.000000
                           std     0.000000
accuracy_with_tol_3        mean    0.000000
                           std     0.000000
accuracy_with_tol_4        mean    0.000000
                           std     0.000000
average_levenshtein        mean    3.797715
                           std     0.129315
average_similarity         mean    0.000000
                           std     0.000000
average_levenshtein_match  mean    0.000000
                           std     0.000000
average_similarity_match   mean 

DeepSeek-R1-Llama-8B-prompt0-0shot - Specific Metrics:


Field/Entity                    HouseNumber  StreetName  Neighborhood  \
Metric                                                                  
accuracy                  mean     0.000000    0.000000      0.000000   
                          std      0.000000    0.000000      0.000000   
precision                 mean     0.000000    0.000000      0.000000   
                          std      0.000000    0.000000      0.000000   
recall                    mean     0.000000    0.000000      0.000000   
                          std      0.000000    0.000000      0.000000   
f1                        mean     0.000000    0.000000      0.000000   
                          std      0.000000    0.000000      0.000000   
accuracy_with_tol_1       mean     0.000000    0.000000      0.000000   
                          std      0.000000    0.000000      0.000000   
accuracy_with_tol_2       mean     0.000000    0.000000      0.000000   
                          std      0.000000    0.000000      0.000000   
accuracy_with_tol_3       mean     0.000000    0.000000      0.000000   
                          std      0.000000    0.000000      0.000000   
accuracy_with_tol_4       mean     0.000000    0.000000      0.000000   
                          std      0.000000    0.000000      0.000000   
average_levenshtein       mean     0.951393    4.820125      0.885421   
                          std      0.120794    0.466509      0.285546   
average_similarity        mean     0.000000    0.000000      0.000000   
                          std      0.000000    0.000000      0.000000   
average_levenshtein_match mean     0.000000    0.000000      0.000000   
                          std      0.000000    0.000000      0.000000   
average_similarity_match  mean     0.000000    0.000000      0.000000   
                          std      0.000000    0.000000      0.000000   
no_match_rate             mean     1.000000    1.000000      1.000000   
                          std      0.000000    0.000000      0.000000   

Field/Entity                        City     State  District   Country  \
Metric                                                                   
accuracy                  mean  0.000000  0.000000  0.000000  0.000000   
                          std   0.000000  0.000000  0.000000  0.000000   
precision                 mean  0.000000  0.000000  0.000000  0.000000   
                          std   0.000000  0.000000  0.000000  0.000000   
recall                    mean  0.000000  0.000000  0.000000  0.000000   
                          std   0.000000  0.000000  0.000000  0.000000   
f1                        mean  0.000000  0.000000  0.000000  0.000000   
                          std   0.000000  0.000000  0.000000  0.000000   
accuracy_with_tol_1       mean  0.000000  0.000000  0.000000  0.000000   
                          std   0.000000  0.000000  0.000000  0.000000   
accuracy_with_tol_2       mean  0.000000  0.000000  0.000000  0.000000   
                          std   0.000000  0.000000  0.000000  0.000000   
accuracy_with_tol_3       mean  0.000000  0.000000  0.000000  0.000000   
                          std   0.000000  0.000000  0.000000  0.000000   
accuracy_with_tol_4       mean  0.000000  0.000000  0.000000  0.000000   
                          std   0.000000  0.000000  0.000000  0.000000   
average_levenshtein       mean  7.784245  0.440834  0.863286  1.635096   
                          std   0.428326  0.143676  0.318076  0.280308   
average_similarity        mean  0.000000  0.000000  0.000000  0.000000   
                          std   0.000000  0.000000  0.000000  0.000000   
average_levenshtein_match mean  0.000000  0.000000  0.000000  0.000000   
                          std   0.000000  0.000000  0.000000  0.000000   
average_similarity_match  mean  0.000000  0.000000  0.000000  0.000000   
                          std   0.000000  0.000000  0.000000  0.000000   
no_match_rate             mean 

In [72]:
supported_entities = [
    "HouseNumber",
    "StreetName",
    "Neighborhood",
    "City",
    "State",
    "District",
    "Country"
]

with FoldEvaluator("Qwen3.5-9B-prompt0-0shot", supported_entities) as evaluator:
    folds_to_process = evaluator.load_preds_and_skip(folds)
    if folds_to_process:
        model = llm_parsers.QwenAddressParsingModel(
            model_name="Qwen/Qwen3.5-9B", 
            device=device,
            example_strategy=llm_parsers.ZeroShot(),
            prompt=prompt0_template
        )
        for fold in tqdm(folds_to_process, unit="fold"):
            evaluator.run_on_fold(fold, model)
        del model

Fold 0 already has predictions for Qwen3.5-9B-prompt0-0shot, loading from file.
Fold 1 already has predictions for Qwen3.5-9B-prompt0-0shot, loading from file.
Fold 2 already has predictions for Qwen3.5-9B-prompt0-0shot, loading from file.
Fold 3 already has predictions for Qwen3.5-9B-prompt0-0shot, loading from file.
Fold 4 already has predictions for Qwen3.5-9B-prompt0-0shot, loading from file.
Fold 5 already has predictions for Qwen3.5-9B-prompt0-0shot, loading from file.
Fold 6 already has predictions for Qwen3.5-9B-prompt0-0shot, loading from file.
Fold 7 already has predictions for Qwen3.5-9B-prompt0-0shot, loading from file.
Fold 8 already has predictions for Qwen3.5-9B-prompt0-0shot, loading from file.
Fold 9 already has predictions for Qwen3.5-9B-prompt0-0shot, loading from file.
Qwen3.5-9B-prompt0-0shot - Metrics for Country City Street House:


accuracy                   mean    0.891843
                           std     0.014924
precision                  mean    0.862784
                           std     0.020151
recall                     mean    0.812803
                           std     0.027471
f1                         mean    0.836923
                           std     0.021945
accuracy_with_tol_1        mean    0.897571
                           std     0.013064
accuracy_with_tol_2        mean    0.903528
                           std     0.012588
accuracy_with_tol_3        mean    0.913834
                           std     0.012659
accuracy_with_tol_4        mean    0.923916
                           std     0.012122
average_levenshtein        mean    0.760525
                           std     0.114467
average_similarity         mean    0.915235
                           std     0.011057
average_levenshtein_match  mean    0.812657
                           std     0.129683
average_similarity_match   mean 

Qwen3.5-9B-prompt0-0shot - Specific Metrics:


Field/Entity                    HouseNumber  StreetName  Neighborhood  \
Metric                                                                  
accuracy                  mean     0.950509    0.912002      0.898257   
                          std      0.017390    0.019937      0.020949   
precision                 mean     0.884387    0.818122      0.370302   
                          std      0.048855    0.044356      0.138067   
recall                    mean     0.888361    0.815920      0.447172   
                          std      0.042780    0.061222      0.155599   
f1                        mean     0.886249    0.816665      0.396624   
                          std      0.044668    0.050443      0.146319   
accuracy_with_tol_1       mean     0.953261    0.923011      0.900092   
                          std      0.015833    0.017912      0.021815   
accuracy_with_tol_2       mean     0.972502    0.924846      0.905588   
                          std      0.017299    0.015435      0.020778   
accuracy_with_tol_3       mean     0.983495    0.929425      0.906505   
                          std      0.014861    0.016200      0.021547   
accuracy_with_tol_4       mean     0.989917    0.940417      0.909249   
                          std      0.008034    0.013865      0.022275   
average_levenshtein       mean     0.175997    0.769817      0.896397   
                          std      0.091335    0.225715      0.300233   
average_similarity        mean     0.962611    0.945106      0.903976   
                          std      0.014900    0.014406      0.021745   
average_levenshtein_match mean     0.180613    0.800556      0.989371   
                          std      0.096019    0.244389      0.360743   
average_similarity_match  mean     0.983315    0.979263      0.990275   
                          std      0.007383    0.009513      0.008361   
no_match_rate             mean     0.021076    0.034812      0.087073   
                          std      0.011468    0.016571      0.023770   

Field/Entity                        City     State  District   Country  \
Metric                                                                   
accuracy                  mean  0.778182  0.940425  0.872594  0.926681   
                          std   0.043650  0.016872  0.030102  0.029625   
precision                 mean  0.866897  0.554557  0.302321  0.882320   
                          std   0.040673  0.094994  0.134037  0.058416   
recall                    mean  0.787264  0.898254  0.357924  0.790785   
                          std   0.039917  0.100547  0.163862  0.083454   
f1                        mean  0.824851  0.680142  0.324329  0.833342   
                          std   0.036882  0.078152  0.138883  0.069991   
accuracy_with_tol_1       mean  0.781835  0.942260  0.875346  0.932177   
                          std   0.041296  0.014986  0.029382  0.025644   
accuracy_with_tol_2       mean  0.782752  0.944095  0.878090  0.934012   
                          std   0.039734  0.016408  0.030913  0.023980   
accuracy_with_tol_3       mean  0.791927  0.946839  0.881760  0.950492   
                          std   0.038973  0.017186  0.029474  0.020854   
accuracy_with_tol_4       mean  0.801993  0.949591  0.894595  0.963336   
                          std   0.040261  0.014489  0.032709  0.016750   
average_levenshtein       mean  1.731476  0.461043  0.953203  0.364812   
                          std   0.335463  0.151048  0.247022  0.143186   
average_similarity        mean  0.816579  0.943214  0.885442  0.936644   
                          std   0.035204  0.014542  0.021492  0.024919   
average_levenshtein_match mean  2.021487  0.489018  1.059525  0.390481   
                          std   0.456389  0.165468  0.284731  0.161270   
average_similarity_match  mean  0.946859  0.997115  0.981676  0.994019   
                          std   0.015207  0.003483  0.013419  0.004944   
no_match_rate             mean 

In [73]:
prompt_llama_optuna_best = llm_parsers.JsonDictPromptTemplate(Path("prompts/optuna_best/best_llama_prompt.txt").read_text())

supported_entities = ["HouseNumber", "StreetName", "City", "State", "Country"]
n_examples = 14
embedding_model = "multi-qa-distilbert-cos-v1"
similarity_threshold = 0.05


with FoldEvaluator("Llama-3-8B-model-best", supported_entities) as evaluator:
    folds_to_process = evaluator.load_preds_and_skip(folds)
    if folds_to_process:
        model = llm_parsers.LlamaAddressParsingModel(
            model_name="meta-llama/Meta-Llama-3-8B-Instruct", 
            device=device,
            example_strategy=llm_parsers.ZeroShot(),
            prompt=prompt_llama_optuna_best
        )
        for fold in tqdm(folds_to_process, unit="fold"):
            embedding_similarity = llm_parsers.SimilarExamples(
                embedding_model=embedding_model,
                example_addresses=fold.train_data['FullAddress'].reset_index(drop=True),
                example_labels=fold.train_data,
                labels_to_include=supported_entities,
                num_examples=n_examples,
                similarity_threshold=similarity_threshold
            )
            model.example_strategy = embedding_similarity
            evaluator.run_on_fold(fold, model)
            model.example_strategy = None # Clear example strategy to save memory
            del embedding_similarity
        del model

Fold 0 already has predictions for Llama-3-8B-model-best, loading from file.
Fold 1 already has predictions for Llama-3-8B-model-best, loading from file.
Fold 2 already has predictions for Llama-3-8B-model-best, loading from file.
Fold 3 already has predictions for Llama-3-8B-model-best, loading from file.
Fold 4 already has predictions for Llama-3-8B-model-best, loading from file.
Fold 5 already has predictions for Llama-3-8B-model-best, loading from file.
Fold 6 already has predictions for Llama-3-8B-model-best, loading from file.
Fold 7 already has predictions for Llama-3-8B-model-best, loading from file.
Fold 8 already has predictions for Llama-3-8B-model-best, loading from file.
Fold 9 already has predictions for Llama-3-8B-model-best, loading from file.
Llama-3-8B-model-best - Metrics for Country City Street House:


accuracy                   mean    0.929658
                           std     0.016857
precision                  mean    0.881475
                           std     0.025320
recall                     mean    0.909745
                           std     0.018516
f1                         mean    0.895324
                           std     0.020884
accuracy_with_tol_1        mean    0.934925
                           std     0.016354
accuracy_with_tol_2        mean    0.939960
                           std     0.014590
accuracy_with_tol_3        mean    0.945692
                           std     0.012839
accuracy_with_tol_4        mean    0.950732
                           std     0.013390
average_levenshtein        mean    0.507571
                           std     0.106908
average_similarity         mean    0.945616
                           std     0.014434
average_levenshtein_match  mean    0.528926
                           std     0.117386
average_similarity_match   mean 

Llama-3-8B-model-best - Specific Metrics:


Field/Entity                    HouseNumber  StreetName      City     State  \
Metric                                                                        
accuracy                  mean     0.953253    0.937681  0.877198  0.955096   
                          std      0.015263    0.026205  0.037161  0.020465   
precision                 mean     0.891869    0.896555  0.880300  0.642873   
                          std      0.033885    0.044774  0.039737  0.124567   
recall                    mean     0.921581    0.883415  0.904443  0.870913   
                          std      0.028038    0.062160  0.027661  0.088650   
f1                        mean     0.906428    0.889561  0.892040  0.734074   
                          std      0.030335    0.050864  0.032040  0.100377   
accuracy_with_tol_1       mean     0.957840    0.941351  0.883603  0.957848   
                          std      0.013105    0.023722  0.036924  0.021242   
accuracy_with_tol_2       mean     0.973411    0.944095  0.884512  0.961510   
                          std      0.009140    0.020918  0.036738  0.020635   
accuracy_with_tol_3       mean     0.980751    0.948682  0.889099  0.963344   
                          std      0.009125    0.018907  0.039484  0.020721   
accuracy_with_tol_4       mean     0.987164    0.951435  0.891852  0.971593   
                          std      0.009866    0.017269  0.039321  0.018560   
average_levenshtein       mean     0.187898    0.604971  0.986147  0.304279   
                          std      0.069624    0.176276  0.301623  0.160984   
average_similarity        mean     0.962183    0.952211  0.910990  0.957137   
                          std      0.012530    0.018785  0.029403  0.021344   
average_levenshtein_match mean     0.193793    0.630994  1.040631  0.320480   
                          std      0.073230    0.188474  0.335729  0.177307   
average_similarity_match  mean     0.989383    0.990340  0.955629  0.998339   
                          std      0.004211    0.007144  0.017159  0.002738   
no_match_rate             mean     0.027490    0.038490  0.046731  0.041234   
                          std      0.012205    0.018228  0.024591  0.022954   

Field/Entity                     Country  ApplicantCurrentAddress  \
Metric                                                              
accuracy                  mean  0.950500                 0.908523   
                          std   0.021719                 0.026394   
precision                 mean  0.856809                 0.902054   
                          std   0.048488                 0.022733   
recall                    mean  0.950157                 0.908326   
                          std   0.029072                 0.030306   
f1                        mean  0.900637                 0.905075   
                          std   0.035963                 0.024774   
accuracy_with_tol_1       mean  0.956906                 0.915812   
                          std   0.021666                 0.023578   
accuracy_with_tol_2       mean  0.957823                 0.922809   
                          std   0.022984                 0.021926   
accuracy_with_tol_3       mean  0.964237                 0.929603   
                          std   0.022257                 0.021353   
accuracy_with_tol_4       mean  0.972477                 0.936037   
                          std   0.019341                 0.018819   
average_levenshtein       mean  0.251268                 0.738774   
                          std   0.137370                 0.211910   
average_similarity        mean  0.957082                 0.932958   
                          std   0.021862                 0.020482   
average_levenshtein_match mean  0.263858                 0.770216   
                          std   0.148767                 0.227422   
average_similarity_match  mean  0.995380                 0.969767   
                          std   0.005746                 0.010

In [74]:
prompt_mistral_optuna_best = llm_parsers.JSONTuplesPromptTemplate(Path("prompts/optuna_best/best_mistral_prompt.txt").read_text())

supported_entities = ["HouseNumber", "StreetName", "City", "District", "State", "Region", "Country"]
n_examples = 13
embedding_model = "multi-qa-mpnet-base-dot-v1"
similarity_threshold = 0.5


with FoldEvaluator("Mistral-3-8B-Instruct-model-best", supported_entities) as evaluator:
    folds_to_process = evaluator.load_preds_and_skip(folds)
    if folds_to_process:
        model = llm_parsers.MistralAddressParsingModel(
            model_name="mistralai/Ministral-3-8B-Instruct-2512", 
            device=device,
            example_strategy=llm_parsers.ZeroShot(),
            prompt=prompt_mistral_optuna_best
        )
        for fold in tqdm(folds_to_process, unit="fold"):
            embedding_similarity = llm_parsers.SimilarExamples(
                embedding_model=embedding_model,
                example_addresses=fold.train_data['FullAddress'].reset_index(drop=True),
                example_labels=fold.train_data,
                labels_to_include=supported_entities,
                num_examples=n_examples,
                similarity_threshold=similarity_threshold
            )
            model.example_strategy = embedding_similarity
            evaluator.run_on_fold(fold, model)
            model.example_strategy = None # Clear example strategy to save memory
            del embedding_similarity
        del model

Fold 0 already has predictions for Mistral-3-8B-Instruct-model-best, loading from file.
Fold 1 already has predictions for Mistral-3-8B-Instruct-model-best, loading from file.
Fold 2 already has predictions for Mistral-3-8B-Instruct-model-best, loading from file.
Fold 3 already has predictions for Mistral-3-8B-Instruct-model-best, loading from file.
Fold 4 already has predictions for Mistral-3-8B-Instruct-model-best, loading from file.
Fold 5 already has predictions for Mistral-3-8B-Instruct-model-best, loading from file.
Fold 6 already has predictions for Mistral-3-8B-Instruct-model-best, loading from file.
Fold 7 already has predictions for Mistral-3-8B-Instruct-model-best, loading from file.
Fold 8 already has predictions for Mistral-3-8B-Instruct-model-best, loading from file.
Fold 9 already has predictions for Mistral-3-8B-Instruct-model-best, loading from file.
Mistral-3-8B-Instruct-model-best - Metrics for Country City Street House:


accuracy                   mean    0.935384
                           std     0.011150
precision                  mean    0.907837
                           std     0.012728
recall                     mean    0.893937
                           std     0.017528
f1                         mean    0.900767
                           std     0.013039
accuracy_with_tol_1        mean    0.940880
                           std     0.011380
accuracy_with_tol_2        mean    0.945690
                           std     0.010770
accuracy_with_tol_3        mean    0.952104
                           std     0.009838
accuracy_with_tol_4        mean    0.957146
                           std     0.010883
average_levenshtein        mean    0.463807
                           std     0.103336
average_similarity         mean    0.952435
                           std     0.010239
average_levenshtein_match  mean    0.479943
                           std     0.110848
average_similarity_match   mean 

Mistral-3-8B-Instruct-model-best - Specific Metrics:


Field/Entity                    HouseNumber  StreetName      City  District  \
Metric                                                                        
accuracy                  mean     0.948666    0.934921  0.878090  0.930334   
                          std      0.017951    0.030106  0.015039  0.015737   
precision                 mean     0.892083    0.891015  0.908953  0.654065   
                          std      0.037531    0.038877  0.025979  0.083673   
recall                    mean     0.882855    0.877255  0.891157  0.616119   
                          std      0.050067    0.062164  0.013471  0.122252   
f1                        mean     0.887274    0.883735  0.899830  0.629205   
                          std      0.042535    0.049093  0.016690  0.081115   
accuracy_with_tol_1       mean     0.951418    0.938590  0.886330  0.930334   
                          std      0.017334    0.027020  0.016941  0.015737   
accuracy_with_tol_2       mean     0.968824    0.940425  0.886330  0.930334   
                          std      0.019450    0.025666  0.016941  0.015737   
accuracy_with_tol_3       mean     0.979817    0.946847  0.892744  0.931251   
                          std      0.018752    0.023179  0.017941  0.017452   
accuracy_with_tol_4       mean     0.989908    0.950500  0.898249  0.936756   
                          std      0.013295    0.022974  0.017596  0.016437   
average_levenshtein       mean     0.184304    0.645304  0.937698  0.568240   
                          std      0.105524    0.244413  0.159697  0.157631   
average_similarity        mean     0.961845    0.950537  0.911135  0.933050   
                          std      0.013514    0.025356  0.011618  0.014002   
average_levenshtein_match mean     0.188228    0.677546  0.997601  0.608232   
                          std      0.108584    0.272293  0.173659  0.173959   
average_similarity_match  mean     0.979815    0.989534  0.968903  0.996021   
                          std      0.010587    0.005244  0.008771  0.003755   
no_match_rate             mean     0.018324    0.039408  0.059591  0.063253   
                          std      0.010570    0.025225  0.011700  0.011027   

Field/Entity                       State    Region   Country  \
Metric                                                         
accuracy                  mean  0.986247  0.941318  0.979858   
                          std   0.006494  0.019954  0.016567   
precision                 mean  0.951996  0.731012  0.950317   
                          std   0.064960  0.143077  0.045789   
recall                    mean  0.858413  0.703479  0.946713   
                          std   0.096477  0.115882  0.045560   
f1                        mean  0.898233  0.709373  0.948118   
                          std   0.050837  0.098386  0.041322   
accuracy_with_tol_1       mean  0.987164  0.944987  0.987181   
                          std   0.006420  0.016224  0.010731   
accuracy_with_tol_2       mean  0.987164  0.951410  0.987181   
                          std   0.006420  0.011522  0.010731   
accuracy_with_tol_3       mean  0.988999  0.953244  0.989008   
                          std   0.008433  0.012602  0.009461   
accuracy_with_tol_4       mean  0.995413  0.964245  0.989925   
                          std   0.006487  0.011005  0.009107   
average_levenshtein       mean  0.083453  0.360367  0.087923   
                          std   0.081559  0.122670  0.074888   
average_similarity        mean  0.986981  0.946595  0.986223   
                          std   0.006329  0.014358  0.011433   
average_levenshtein_match mean  0.085009  0.380656  0.089477   
                          std   0.083816  0.133725  0.076500   
average_similarity_match  mean  0.999815  0.995901  0.997162   
                          std   0.000586  0.004569  0.003375   
no_match_rate             mean  0.012836  0.049516  0.010984   
                          std   0.006420  0.013165  0.00942

In [75]:
prompt_deepseek_optuna_best = llm_parsers.JsonDictPromptTemplate(Path("prompts/optuna_best/best_deepseek_prompt.txt").read_text())

supported_entities = ['HouseNumber', 'StreetName', 'City', 'District', 'Region', 'Country']
n_examples = 8
embedding_model = "multi-qa-mpnet-base-dot-v1"
similarity_threshold = 0.3


with FoldEvaluator("DeepSeek-R1-Llama-8B-model-best", supported_entities) as evaluator:
    folds_to_process = evaluator.load_preds_and_skip(folds)
    if folds_to_process:
        model = llm_parsers.DeepSeekAddressParsingModel(
            model_name="deepseek-ai/DeepSeek-R1-Distill-Llama-8B", 
            device=device,
            example_strategy=llm_parsers.ZeroShot(),
            prompt=prompt_deepseek_optuna_best
        )
        for fold in tqdm(folds_to_process, unit="fold"):
            pattern_similarity = llm_parsers.NERPatternSimilarExamples(
                example_addresses=fold.train_data['FullAddress'].reset_index(drop=True),
                example_labels=fold.train_data,
                labels_to_include=supported_entities,
                num_examples=n_examples,
                model_dir = fold.models_path / "ner_bzk"
            )
            embedding_similarity = llm_parsers.SimilarExamples(
                embedding_model=embedding_model,
                example_addresses=fold.train_data['FullAddress'].reset_index(drop=True),
                example_labels=fold.train_data,
                labels_to_include=supported_entities,
                num_examples=n_examples,
                similarity_threshold=similarity_threshold
            )
            hybrid_similarity = llm_parsers.HybridSimilarExamples(
                pattern_strategy=pattern_similarity,
                embedding_strategy=embedding_similarity,
                num_examples=n_examples,
                pool_size=n_examples
            )
            model.example_strategy = hybrid_similarity
            evaluator.run_on_fold(fold, model)
            model.example_strategy = None # Clear example strategy to save memory
            del hybrid_similarity, embedding_similarity, pattern_similarity
        del model

Fold 0 already has predictions for DeepSeek-R1-Llama-8B-model-best, loading from file.
Fold 1 already has predictions for DeepSeek-R1-Llama-8B-model-best, loading from file.
Fold 2 already has predictions for DeepSeek-R1-Llama-8B-model-best, loading from file.
Fold 3 already has predictions for DeepSeek-R1-Llama-8B-model-best, loading from file.
Fold 4 already has predictions for DeepSeek-R1-Llama-8B-model-best, loading from file.
Fold 5 already has predictions for DeepSeek-R1-Llama-8B-model-best, loading from file.
Fold 6 already has predictions for DeepSeek-R1-Llama-8B-model-best, loading from file.
Fold 7 already has predictions for DeepSeek-R1-Llama-8B-model-best, loading from file.
Fold 8 already has predictions for DeepSeek-R1-Llama-8B-model-best, loading from file.
Fold 9 already has predictions for DeepSeek-R1-Llama-8B-model-best, loading from file.
DeepSeek-R1-Llama-8B-model-best - Metrics for Country City Street House:


accuracy                   mean     0.003899
                           std      0.006842
precision                  mean     0.300000
                           std      0.483046
recall                     mean     0.001370
                           std      0.002208
f1                         mean     0.002727
                           std      0.004395
accuracy_with_tol_1        mean     0.003899
                           std      0.006842
accuracy_with_tol_2        mean     0.003899
                           std      0.006842
accuracy_with_tol_3        mean     0.004587
                           std      0.007797
accuracy_with_tol_4        mean     0.007569
                           std      0.012932
average_levenshtein        mean     3.793357
                           std      0.127809
average_similarity         mean     0.003899
                           std      0.006842
average_levenshtein_match  mean    53.850000
                           std     97.666238
average_si

DeepSeek-R1-Llama-8B-model-best - Specific Metrics:


Field/Entity                    HouseNumber  StreetName       City  District  \
Metric                                                                         
accuracy                  mean     0.000000    0.000000   0.015596  0.000000   
                          std      0.000000    0.000000   0.027370  0.000000   
precision                 mean     0.000000    0.000000   0.300000  0.000000   
                          std      0.000000    0.000000   0.483046  0.000000   
recall                    mean     0.000000    0.000000   0.002877  0.000000   
                          std      0.000000    0.000000   0.004633  0.000000   
f1                        mean     0.000000    0.000000   0.005698  0.000000   
                          std      0.000000    0.000000   0.009178  0.000000   
accuracy_with_tol_1       mean     0.000000    0.000000   0.015596  0.000000   
                          std      0.000000    0.000000   0.027370  0.000000   
accuracy_with_tol_2       mean     0.000000    0.000000   0.015596  0.000000   
                          std      0.000000    0.000000   0.027370  0.000000   
accuracy_with_tol_3       mean     0.000000    0.000000   0.018349  0.000000   
                          std      0.000000    0.000000   0.031187  0.000000   
accuracy_with_tol_4       mean     0.000000    0.000000   0.030275  0.000000   
                          std      0.000000    0.000000   0.051726  0.000000   
average_levenshtein       mean     0.951393    4.820125   7.766814  0.863286   
                          std      0.120794    0.466509   0.418823  0.318076   
average_similarity        mean     0.000000    0.000000   0.015596  0.000000   
                          std      0.000000    0.000000   0.027370  0.000000   
average_levenshtein_match mean     0.000000    0.000000  53.850000  0.000000   
                          std      0.000000    0.000000  97.666238  0.000000   
average_similarity_match  mean     0.000000    0.000000   0.300000  0.000000   
                          std      0.000000    0.000000   0.483046  0.000000   
no_match_rate             mean     1.000000    1.000000   0.984404  1.000000   
                          std      0.000000    0.000000   0.027370  0.000000   

Field/Entity                      Region   Country  ApplicantCurrentAddress  \
Metric                                                                        
accuracy                  mean  0.087156  0.000000                 0.000735   
                          std   0.275611  0.000000                 0.002325   
precision                 mean  0.000000  0.000000                 0.000000   
                          std   0.000000  0.000000                 0.000000   
recall                    mean  0.000000  0.000000                 0.000000   
                          std   0.000000  0.000000                 0.000000   
f1                        mean  0.000000  0.000000                 0.000000   
                          std   0.000000  0.000000                 0.000000   
accuracy_with_tol_1       mean  0.088991  0.000000                 0.000735   
                          std   0.281414  0.000000                 0.002325   
accuracy_with_tol_2       mean  0.091743  0.000000                 0.000735   
                          std   0.290117  0.000000                 0.002325   
accuracy_with_tol_3       mean  0.091743  0.000000                 0.001345   
                          std   0.290117  0.000000                 0.002851   
accuracy_with_tol_4       mean  0.094495  0.000000                 0.004140   
                          std   0.298821  0.000000                 0.007564   
average_levenshtein       mean  0.652652  1.635096                 5.903988   
                          std   0.110475  0.280308                 0.272232   
average_similarity        mean  0.087920  0.000000                 0.000735   
                          std   0.278029  0.000000                 0.002325   
average_levenshtein_m

In [76]:
prompt_qwen_optuna_best = llm_parsers.JsonDictPromptTemplate(Path("prompts/optuna_best/best_qwen_prompt.txt").read_text())

supported_entities = ["HouseNumber", "StreetName", "Neighborhood", "City", "Country"]
n_examples = 15
embedding_model = "all-MiniLM-L6-v2"
similarity_threshold = 0.35


with FoldEvaluator("Qwen3.5-9B-best-from-optuna", supported_entities) as evaluator:
    folds_to_process = evaluator.load_preds_and_skip(folds)
    if folds_to_process:
        model = llm_parsers.QwenAddressParsingModel(
            model_name="Qwen/Qwen3.5-9B", 
            device=device,
            example_strategy=llm_parsers.ZeroShot(),
            prompt=prompt_qwen_optuna_best
        )
        for fold in tqdm(folds_to_process, unit="fold"):
            pattern_similarity = llm_parsers.NERPatternSimilarExamples(
                example_addresses=fold.train_data['FullAddress'].reset_index(drop=True),
                example_labels=fold.train_data,
                labels_to_include=supported_entities,
                num_examples=n_examples,
                model_dir = fold.models_path / "ner_bzk"
            )
            embedding_similarity = llm_parsers.SimilarExamples(
                embedding_model=embedding_model,
                example_addresses=fold.train_data['FullAddress'].reset_index(drop=True),
                example_labels=fold.train_data,
                labels_to_include=supported_entities,
                num_examples=n_examples,
                similarity_threshold=similarity_threshold
            )
            hybrid_similarity = llm_parsers.HybridSimilarExamples(
                pattern_strategy=pattern_similarity,
                embedding_strategy=embedding_similarity,
                num_examples=n_examples,
                pool_size=n_examples
            )
            model.example_strategy = hybrid_similarity
            evaluator.run_on_fold(fold, model)
            model.example_strategy = None # Clear example strategy to save memory
            del hybrid_similarity, embedding_similarity, pattern_similarity
        del model

Fold 0 already has predictions for Qwen3.5-9B-best-from-optuna, loading from file.
Fold 1 already has predictions for Qwen3.5-9B-best-from-optuna, loading from file.
Fold 2 already has predictions for Qwen3.5-9B-best-from-optuna, loading from file.
Fold 3 already has predictions for Qwen3.5-9B-best-from-optuna, loading from file.
Fold 4 already has predictions for Qwen3.5-9B-best-from-optuna, loading from file.
Fold 5 already has predictions for Qwen3.5-9B-best-from-optuna, loading from file.
Fold 6 already has predictions for Qwen3.5-9B-best-from-optuna, loading from file.
Fold 7 already has predictions for Qwen3.5-9B-best-from-optuna, loading from file.
Fold 8 already has predictions for Qwen3.5-9B-best-from-optuna, loading from file.
Fold 9 already has predictions for Qwen3.5-9B-best-from-optuna, loading from file.
Qwen3.5-9B-best-from-optuna - Metrics for Country City Street House:


accuracy                   mean    0.936314
                           std     0.013161
precision                  mean    0.894010
                           std     0.020701
recall                     mean    0.905084
                           std     0.016348
f1                         mean    0.899481
                           std     0.017808
accuracy_with_tol_1        mean    0.941580
                           std     0.012394
accuracy_with_tol_2        mean    0.947308
                           std     0.010997
accuracy_with_tol_3        mean    0.954633
                           std     0.008742
accuracy_with_tol_4        mean    0.958761
                           std     0.009439
average_levenshtein        mean    0.441874
                           std     0.094128
average_similarity         mean    0.952953
                           std     0.011021
average_levenshtein_match  mean    0.455833
                           std     0.101747
average_similarity_match   mean 

Qwen3.5-9B-best-from-optuna - Specific Metrics:


Field/Entity                    HouseNumber  StreetName  Neighborhood  \
Metric                                                                  
accuracy                  mean     0.955104    0.937690      0.902844   
                          std      0.013202    0.022730      0.024906   
precision                 mean     0.890676    0.878055      0.416325   
                          std      0.036791    0.053167      0.154575   
recall                    mean     0.896564    0.861443      0.592338   
                          std      0.035742    0.055976      0.203015   
f1                        mean     0.893499    0.869505      0.478791   
                          std      0.034710    0.053185      0.171298   
accuracy_with_tol_1       mean     0.959683    0.942277      0.906505   
                          std      0.011535    0.019739      0.021977   
accuracy_with_tol_2       mean     0.978007    0.944112      0.912927   
                          std      0.012372    0.018458      0.021714   
accuracy_with_tol_3       mean     0.989908    0.952352      0.913845   
                          std      0.010984    0.016574      0.022115   
accuracy_with_tol_4       mean     0.995413    0.956939      0.917515   
                          std      0.006487    0.014922      0.022849   
average_levenshtein       mean     0.142068    0.547815      0.781768   
                          std      0.062849    0.200643      0.134838   
average_similarity        mean     0.968867    0.959464      0.905985   
                          std      0.009322    0.011354      0.024436   
average_levenshtein_match mean     0.144084    0.562362      0.861003   
                          std      0.064452    0.209849      0.170780   
average_similarity_match  mean     0.980552    0.982891      0.993388   
                          std      0.006934    0.008469      0.003481   
no_match_rate             mean     0.011910    0.023812      0.087990   
                          std      0.007537    0.010674      0.024144   

Field/Entity                        City   Country  ApplicantCurrentAddress  \
Metric                                                                        
accuracy                  mean  0.894604  0.957857                 0.899828   
                          std   0.029674  0.021209                 0.027592   
precision                 mean  0.901514  0.896205                 0.894202   
                          std   0.033666  0.050927                 0.031150   
recall                    mean  0.916456  0.938206                 0.892163   
                          std   0.019656  0.034406                 0.028948   
f1                        mean  0.908834  0.916215                 0.893131   
                          std   0.026112  0.038093                 0.029241   
accuracy_with_tol_1       mean  0.901009  0.963353                 0.907820   
                          std   0.029581  0.019289                 0.025435   
accuracy_with_tol_2       mean  0.901009  0.966105                 0.920163   
                          std   0.029581  0.018311                 0.022298   
accuracy_with_tol_3       mean  0.904679  0.971593                 0.933631   
                          std   0.031823  0.016982                 0.017974   
accuracy_with_tol_4       mean  0.904679  0.978015                 0.942364   
                          std   0.031823  0.016264                 0.018579   
average_levenshtein       mean  0.839483  0.238132                 0.672895   
                          std   0.236118  0.166631                 0.198111   
average_similarity        mean  0.920630  0.962852                 0.929426   
                          std   0.026836  0.020095                 0.022281   
average_levenshtein_match mean  0.882118  0.249998                 0.702007   
                          std   0.264726  0.179119                 0.219468   
average_similarity_match  mean  0.962083  0.998528         

In [77]:
table_label_remapping = {
    # Model names
    "libpostal": "Libpostal",
    "deepparse": "DeepParse",
    "xlm-roberta-large": "XLM-RoBERTa",
    "SpaCy": "SpaCy",
    "Qwen3.5-9B-prompt0-0shot": "Qwen (1st p. 0shot)",
    "Llama-3-8B-model-best": "Llama (Optuna)",
    "Mistral-3-8B-Instruct-model-best": "Mistral (Optuna)",
    "DeepSeek-R1-Llama-8B-model-best": "DeepSeek (Optuna)",
    "Qwen3.5-9B-best-from-optuna": "Qwen (Optuna)",
    # Column names
    "precision": "Precision",
    "recall": "Recall",
    "accuracy": "Accuracy",
    #"f1": "F$_1$-Score" # pandas will escape latex, so do this after generating the table
    "HouseNumber": "House",
    "StreetName": "Street",
    "Neighborhood": "Neigh.",
    "ApplicantCurrentAddress": "App. Addr.",
    "ApplicantBirthPlace": "App. BP",
    # Term "Victim" is deprecated in the code, text uses "Persecutee" 
    "VictimCurrentAddress": "Pers. Addr.",
    "VictimBirthPlace": "Pers. BP",
    "VictimDeathPlace": "Pers. DP"
}

def style_results(df : pd.DataFrame):
    styled_df = df.apply(
        lambda row: pd.Series({
            col : "-" if pd.isna(row[(col, 'mean')]) else f"{row[(col, 'mean')]:.2f}±{row[(col, 'std')]:.2f}"
            for col in row.index.get_level_values(0)
        }, name=row.name),
        axis=1
    )
    styled_df.index.name = "Model"
    styled_df = styled_df.rename(index=lambda l : table_label_remapping.get(l, l)).reset_index()
    styled_df = styled_df.style.apply(
        lambda s: [
            'font-weight: bold' 
            if s.name != "Model" and df[(s.name, 'mean')].iat[k].round(2) == df[(s.name, 'mean')].max().round(2)
            else '' 
            for k in range(len(s))
        ],
        axis=0
    )
    styled_df = styled_df.relabel_index(
        [table_label_remapping.get(l, l) for l in styled_df.columns],
        axis=1
    )
    styled_df = styled_df.apply_index(lambda s: ["font-weight: bold"] * len(s), axis=1)
    styled_df = styled_df.hide(axis="index") # Hide index since we already have a "Model" column
    return styled_df

def print_latex(styled_df) -> str:
    # Extend the LaTeX table to include the standard deviation in the format "mean±std"
    latex_str = styled_df.to_latex(hrules=True, convert_css=True, column_format="l" + "c" * (len(styled_df.columns)))
    latex_str = latex_str.replace(" f1 ", " F$_1$ ")
    latex_str = latex_str.replace("±", "$\\pm$")
    latex_lines = latex_str.splitlines()
    latex_lines[0] += r" % LaTeX table generated in cross_val_evaluation.ipynb"
    latex_str = "\n".join(latex_lines)
    print(latex_str)

In [78]:
all_experiments = pd.concat([
    pd.DataFrame({k : v[["f1", "precision", "recall", "accuracy"]] for k, v in required_results.items()}),
    pd.DataFrame({k : v.unstack(level=1)["f1"] for k, v in specific_results.items()}).loc[["City", "Country", "StreetName"]]
]).T
all_experiments_styled = style_results(all_experiments)

display(all_experiments_styled)

Model,f1,Precision,Recall,Accuracy,City,Country,Street
Libpostal,0.51±0.03,0.64±0.04,0.42±0.03,0.69±0.02,0.41±0.04,0.45±0.09,0.51±0.04
DeepParse,0.31±0.03,0.34±0.03,0.28±0.03,0.58±0.03,0.30±0.05,0.12±0.06,0.06±0.03
XLM-RoBERTa,0.68±0.02,0.71±0.03,0.66±0.02,0.80±0.02,0.57±0.04,0.79±0.04,0.74±0.05
SpaCy,0.80±0.04,0.82±0.04,0.78±0.04,0.87±0.03,0.83±0.04,0.88±0.03,0.70±0.05
Llama-3-8B-prompt0-0shot,0.77±0.03,0.74±0.03,0.79±0.03,0.83±0.02,0.78±0.04,0.75±0.07,0.67±0.06
Mistral-3-8B-Instruct-prompt0-0shot,0.73±0.02,0.77±0.03,0.69±0.03,0.83±0.01,0.72±0.05,0.81±0.07,0.54±0.05
DeepSeek-R1-Llama-8B-prompt0-0shot,0.00±0.00,0.00±0.00,0.00±0.00,0.00±0.00,0.00±0.00,0.00±0.00,0.00±0.00
Qwen (1st p. 0shot),0.84±0.02,0.86±0.02,0.81±0.03,0.89±0.01,0.82±0.04,0.83±0.07,0.82±0.05
Llama (Optuna),0.90±0.02,0.88±0.03,0.91±0.02,0.93±0.02,0.89±0.03,0.90±0.04,0.89±0.05
Mistral (Optuna),0.90±0.01,0.91±0.01,0.89±0.02,0.94±0.01,0.90±0.02,0.95±0.04,0.88±0.05


In [79]:
# Subset of experiments selected for inclusion in the paper.
selected_experiments = all_experiments.drop(index=[
    "DeepSeek-R1-Llama-8B-model-best", "Llama-3-8B-prompt0-0shot",
    "Mistral-3-8B-Instruct-prompt0-0shot", "DeepSeek-R1-Llama-8B-prompt0-0shot"
])
selected_experiments_styled = style_results(selected_experiments)

display(selected_experiments_styled)

Model,f1,Precision,Recall,Accuracy,City,Country,Street
Libpostal,0.51±0.03,0.64±0.04,0.42±0.03,0.69±0.02,0.41±0.04,0.45±0.09,0.51±0.04
DeepParse,0.31±0.03,0.34±0.03,0.28±0.03,0.58±0.03,0.30±0.05,0.12±0.06,0.06±0.03
XLM-RoBERTa,0.68±0.02,0.71±0.03,0.66±0.02,0.80±0.02,0.57±0.04,0.79±0.04,0.74±0.05
SpaCy,0.80±0.04,0.82±0.04,0.78±0.04,0.87±0.03,0.83±0.04,0.88±0.03,0.70±0.05
Qwen (1st p. 0shot),0.84±0.02,0.86±0.02,0.81±0.03,0.89±0.01,0.82±0.04,0.83±0.07,0.82±0.05
Llama (Optuna),0.90±0.02,0.88±0.03,0.91±0.02,0.93±0.02,0.89±0.03,0.90±0.04,0.89±0.05
Mistral (Optuna),0.90±0.01,0.91±0.01,0.89±0.02,0.94±0.01,0.90±0.02,0.95±0.04,0.88±0.05
Qwen (Optuna),0.90±0.02,0.89±0.02,0.91±0.02,0.94±0.01,0.91±0.03,0.92±0.04,0.87±0.05


In [80]:
print_latex(selected_experiments_styled)

\begin{tabular}{lcccccccc} % LaTeX table generated in cross_val_evaluation.ipynb
\toprule
\bfseries Model & \bfseries F$_1$ & \bfseries Precision & \bfseries Recall & \bfseries Accuracy & \bfseries City & \bfseries Country & \bfseries Street \\
\midrule
Libpostal & 0.51$\pm$0.03 & 0.64$\pm$0.04 & 0.42$\pm$0.03 & 0.69$\pm$0.02 & 0.41$\pm$0.04 & 0.45$\pm$0.09 & 0.51$\pm$0.04 \\
DeepParse & 0.31$\pm$0.03 & 0.34$\pm$0.03 & 0.28$\pm$0.03 & 0.58$\pm$0.03 & 0.30$\pm$0.05 & 0.12$\pm$0.06 & 0.06$\pm$0.03 \\
XLM-RoBERTa & 0.68$\pm$0.02 & 0.71$\pm$0.03 & 0.66$\pm$0.02 & 0.80$\pm$0.02 & 0.57$\pm$0.04 & 0.79$\pm$0.04 & 0.74$\pm$0.05 \\
SpaCy & 0.80$\pm$0.04 & 0.82$\pm$0.04 & 0.78$\pm$0.04 & 0.87$\pm$0.03 & 0.83$\pm$0.04 & 0.88$\pm$0.03 & 0.70$\pm$0.05 \\
Qwen (1st p. 0shot) & 0.84$\pm$0.02 & 0.86$\pm$0.02 & 0.81$\pm$0.03 & 0.89$\pm$0.01 & 0.82$\pm$0.04 & 0.83$\pm$0.07 & 0.82$\pm$0.05 \\
Llama (Optuna) & \bfseries 0.90$\pm$0.02 & 0.88$\pm$0.03 & \bfseries 0.91$\pm$0.02 & 0.93$\pm$0.02 & 0.89$\pm$0.0

In [81]:
metric = "f1"
specific_metrics = pd.DataFrame({k : v.unstack(level=1)[metric] for k, v in specific_results.items()}).T


selected_specific_metrics = specific_metrics
# Sanity check that the selected experiments include all the best results
assert all(selected_specific_metrics[c].argmax() == specific_metrics[c].argmax() for c in selected_specific_metrics.columns), "Selected experiments must have the same best metrics as all experiments for the required entities"


entity_specific_metrics = selected_specific_metrics[[c for c in selected_specific_metrics.columns if c[0] in ALL_ENTITIES]]
entity_specific_metrics_styled = style_results(entity_specific_metrics)
display(entity_specific_metrics_styled)
print_latex(entity_specific_metrics_styled)

Model,City,Country,District,House,Neigh.,Region,State,Street
Libpostal,0.41±0.04,0.45±0.09,-,0.72±0.06,-,-,0.50±0.26,0.51±0.04
DeepParse,0.30±0.05,0.12±0.06,-,0.75±0.05,-,-,-,0.06±0.03
XLM-RoBERTa,0.57±0.04,0.79±0.04,-,0.81±0.04,-,-,0.29±0.16,0.74±0.05
SpaCy,0.83±0.04,0.88±0.03,0.55±0.10,0.76±0.06,0.19±0.16,0.00±0.00,0.55±0.20,0.70±0.05
Llama-3-8B-prompt0-0shot,0.78±0.04,0.75±0.07,0.04±0.07,0.86±0.03,0.30±0.11,-,0.57±0.09,0.67±0.06
Mistral-3-8B-Instruct-prompt0-0shot,0.72±0.05,0.81±0.07,0.29±0.13,0.88±0.03,0.43±0.23,-,0.71±0.09,0.54±0.05
DeepSeek-R1-Llama-8B-prompt0-0shot,0.00±0.00,0.00±0.00,0.00±0.00,0.00±0.00,0.00±0.00,-,0.00±0.00,0.00±0.00
Qwen (1st p. 0shot),0.82±0.04,0.83±0.07,0.32±0.14,0.89±0.04,0.40±0.15,-,0.68±0.08,0.82±0.05
Llama (Optuna),0.89±0.03,0.90±0.04,-,0.91±0.03,-,-,0.73±0.10,0.89±0.05
Mistral (Optuna),0.90±0.02,0.95±0.04,0.63±0.08,0.89±0.04,-,0.71±0.10,0.90±0.05,0.88±0.05


\begin{tabular}{lccccccccc} % LaTeX table generated in cross_val_evaluation.ipynb
\toprule
\bfseries Model & \bfseries City & \bfseries Country & \bfseries District & \bfseries House & \bfseries Neigh. & \bfseries Region & \bfseries State & \bfseries Street \\
\midrule
Libpostal & 0.41$\pm$0.04 & 0.45$\pm$0.09 & - & 0.72$\pm$0.06 & - & - & 0.50$\pm$0.26 & 0.51$\pm$0.04 \\
DeepParse & 0.30$\pm$0.05 & 0.12$\pm$0.06 & - & 0.75$\pm$0.05 & - & - & - & 0.06$\pm$0.03 \\
XLM-RoBERTa & 0.57$\pm$0.04 & 0.79$\pm$0.04 & - & 0.81$\pm$0.04 & - & - & 0.29$\pm$0.16 & 0.74$\pm$0.05 \\
SpaCy & 0.83$\pm$0.04 & 0.88$\pm$0.03 & 0.55$\pm$0.10 & 0.76$\pm$0.06 & 0.19$\pm$0.16 & 0.00$\pm$0.00 & 0.55$\pm$0.20 & 0.70$\pm$0.05 \\
Llama-3-8B-prompt0-0shot & 0.78$\pm$0.04 & 0.75$\pm$0.07 & 0.04$\pm$0.07 & 0.86$\pm$0.03 & 0.30$\pm$0.11 & - & 0.57$\pm$0.09 & 0.67$\pm$0.06 \\
Mistral-3-8B-Instruct-prompt0-0shot & 0.72$\pm$0.05 & 0.81$\pm$0.07 & 0.29$\pm$0.13 & 0.88$\pm$0.03 & 0.43$\pm$0.23 & - & 0.71$\pm$0.09 & 0.54$\

In [82]:
field_specific_metrics = selected_specific_metrics[[c for c in selected_specific_metrics.columns if c[0] in BZK_ADDRESS_FIELDS]]
field_specific_metrics_styled = style_results(field_specific_metrics)
display(field_specific_metrics_styled)
print_latex(field_specific_metrics_styled)

Model,App. BP,App. Addr.,Pers. BP,Pers. Addr.,Pers. DP
Libpostal,0.53±0.08,0.50±0.05,0.51±0.13,0.50±0.13,0.48±0.13
DeepParse,0.33±0.08,0.27±0.03,0.46±0.10,0.28±0.08,0.30±0.12
XLM-RoBERTa,0.68±0.08,0.68±0.03,0.68±0.06,0.69±0.10,0.64±0.23
SpaCy,0.88±0.04,0.77±0.05,0.90±0.06,0.73±0.08,0.70±0.15
Llama-3-8B-prompt0-0shot,0.68±0.07,0.82±0.03,0.68±0.14,0.80±0.06,0.61±0.23
Mistral-3-8B-Instruct-prompt0-0shot,0.68±0.08,0.76±0.03,0.72±0.09,0.68±0.10,0.77±0.19
DeepSeek-R1-Llama-8B-prompt0-0shot,0.00±0.00,0.00±0.00,0.00±0.00,0.00±0.00,0.00±0.00
Qwen (1st p. 0shot),0.82±0.05,0.85±0.02,0.82±0.08,0.81±0.08,0.81±0.14
Llama (Optuna),0.90±0.04,0.91±0.02,0.91±0.07,0.86±0.05,0.82±0.14
Mistral (Optuna),0.91±0.05,0.91±0.02,0.93±0.07,0.87±0.05,0.79±0.18


\begin{tabular}{lcccccc} % LaTeX table generated in cross_val_evaluation.ipynb
\toprule
\bfseries Model & \bfseries App. BP & \bfseries App. Addr. & \bfseries Pers. BP & \bfseries Pers. Addr. & \bfseries Pers. DP \\
\midrule
Libpostal & 0.53$\pm$0.08 & 0.50$\pm$0.05 & 0.51$\pm$0.13 & 0.50$\pm$0.13 & 0.48$\pm$0.13 \\
DeepParse & 0.33$\pm$0.08 & 0.27$\pm$0.03 & 0.46$\pm$0.10 & 0.28$\pm$0.08 & 0.30$\pm$0.12 \\
XLM-RoBERTa & 0.68$\pm$0.08 & 0.68$\pm$0.03 & 0.68$\pm$0.06 & 0.69$\pm$0.10 & 0.64$\pm$0.23 \\
SpaCy & 0.88$\pm$0.04 & 0.77$\pm$0.05 & 0.90$\pm$0.06 & 0.73$\pm$0.08 & 0.70$\pm$0.15 \\
Llama-3-8B-prompt0-0shot & 0.68$\pm$0.07 & 0.82$\pm$0.03 & 0.68$\pm$0.14 & 0.80$\pm$0.06 & 0.61$\pm$0.23 \\
Mistral-3-8B-Instruct-prompt0-0shot & 0.68$\pm$0.08 & 0.76$\pm$0.03 & 0.72$\pm$0.09 & 0.68$\pm$0.10 & 0.77$\pm$0.19 \\
DeepSeek-R1-Llama-8B-prompt0-0shot & 0.00$\pm$0.00 & 0.00$\pm$0.00 & 0.00$\pm$0.00 & 0.00$\pm$0.00 & 0.00$\pm$0.00 \\
Qwen (1st p. 0shot) & 0.82$\pm$0.05 & 0.85$\pm$0.02 & 0.82$\

# Qwen Specific tables

In [83]:
qwen_specific_results = (
    specific_results["Qwen3.5-9B-best-from-optuna"].unstack(level=[1, 2])[["f1", "precision", "recall", "accuracy"]]
).T

qwen_entity_results = pd.concat([
    qwen_specific_results[["HouseNumber", "StreetName", "Neighborhood", "City", "Country"]],
    required_results["Qwen3.5-9B-best-from-optuna"][["f1", "precision", "recall", "accuracy"]]
], axis=1).T
qwen_entity_results.rename(index={0: "House, Street, City, Country"}, inplace=True)

qwen_entity_results_styled = style_results(qwen_entity_results)
display(qwen_entity_results_styled)
print_latex(qwen_entity_results_styled)


Model,f1,Precision,Recall,Accuracy
House,0.89±0.03,0.89±0.04,0.90±0.04,0.96±0.01
Street,0.87±0.05,0.88±0.05,0.86±0.06,0.94±0.02
Neigh.,0.48±0.17,0.42±0.15,0.59±0.20,0.90±0.02
City,0.91±0.03,0.90±0.03,0.92±0.02,0.89±0.03
Country,0.92±0.04,0.90±0.05,0.94±0.03,0.96±0.02
"House, Street, City, Country",0.90±0.02,0.89±0.02,0.91±0.02,0.94±0.01


\begin{tabular}{lccccc} % LaTeX table generated in cross_val_evaluation.ipynb
\toprule
\bfseries Model & \bfseries F$_1$ & \bfseries Precision & \bfseries Recall & \bfseries Accuracy \\
\midrule
House & 0.89$\pm$0.03 & 0.89$\pm$0.04 & 0.90$\pm$0.04 & \bfseries 0.96$\pm$0.01 \\
Street & 0.87$\pm$0.05 & 0.88$\pm$0.05 & 0.86$\pm$0.06 & 0.94$\pm$0.02 \\
Neigh. & 0.48$\pm$0.17 & 0.42$\pm$0.15 & 0.59$\pm$0.20 & 0.90$\pm$0.02 \\
City & 0.91$\pm$0.03 & \bfseries 0.90$\pm$0.03 & 0.92$\pm$0.02 & 0.89$\pm$0.03 \\
Country & \bfseries 0.92$\pm$0.04 & \bfseries 0.90$\pm$0.05 & \bfseries 0.94$\pm$0.03 & \bfseries 0.96$\pm$0.02 \\
House, Street, City, Country & 0.90$\pm$0.02 & 0.89$\pm$0.02 & 0.91$\pm$0.02 & 0.94$\pm$0.01 \\
\bottomrule
\end{tabular}


In [84]:
qwen_bzk_field_styled = style_results(qwen_specific_results[[f for f in BZK_ADDRESS_FIELDS]].T.sort_index())
display(qwen_bzk_field_styled)
print_latex(qwen_bzk_field_styled)

Model,f1,Precision,Recall,Accuracy
App. BP,0.91±0.03,0.91±0.03,0.92±0.04,0.96±0.02
App. Addr.,0.89±0.03,0.89±0.03,0.89±0.03,0.90±0.03
Pers. BP,0.94±0.05,0.94±0.04,0.95±0.07,0.98±0.02
Pers. Addr.,0.87±0.05,0.86±0.05,0.88±0.05,0.89±0.04
Pers. DP,0.89±0.09,0.81±0.15,0.99±0.05,0.97±0.03


\begin{tabular}{lccccc} % LaTeX table generated in cross_val_evaluation.ipynb
\toprule
\bfseries Model & \bfseries F$_1$ & \bfseries Precision & \bfseries Recall & \bfseries Accuracy \\
\midrule
App. BP & 0.91$\pm$0.03 & 0.91$\pm$0.03 & 0.92$\pm$0.04 & 0.96$\pm$0.02 \\
App. Addr. & 0.89$\pm$0.03 & 0.89$\pm$0.03 & 0.89$\pm$0.03 & 0.90$\pm$0.03 \\
Pers. BP & \bfseries 0.94$\pm$0.05 & \bfseries 0.94$\pm$0.04 & 0.95$\pm$0.07 & \bfseries 0.98$\pm$0.02 \\
Pers. Addr. & 0.87$\pm$0.05 & 0.86$\pm$0.05 & 0.88$\pm$0.05 & 0.89$\pm$0.04 \\
Pers. DP & 0.89$\pm$0.09 & 0.81$\pm$0.15 & \bfseries 0.99$\pm$0.05 & 0.97$\pm$0.03 \\
\bottomrule
\end{tabular}


In [102]:
def eqna(x, y):
    if pd.isna(x) and pd.isna(y):
        return True
    elif pd.isna(x) or pd.isna(y):
        return False
    else:
        return x.lower() == y.lower()

def display_predictions(split : str, row_idx : int):
    example_idx = (split, row_idx)
    example_address = bzkopen_merged.loc[example_idx, "FullAddress"]
    print(f"Example address ({split} {row_idx}):\n{example_address}")
    example_predictions = []
    for model_name, preds_df in predictions_dfs.items():
        if "DeepSeek" in model_name:
            continue # DeepSeek predictions are malformed
        pred_row = preds_df.loc[example_idx].copy()
        pred_row.name = model_name
        example_predictions.append(pred_row)
    truth_row = bzkopen_merged.loc[example_idx, ALL_ENTITIES].copy().drop(columns="FullAddress")
    truth_row.name = "Ground Truth"
    example_predictions.append(truth_row)
    example_predictions = pd.DataFrame(example_predictions)
    example_predictions.dropna(how="all", axis=1, inplace=True)
    example_predictions.drop(columns=["fullConversation", "___example_metadata"], errors="ignore", inplace=True)
    example_predictions_styled = example_predictions.style.apply(
        lambda s: [
            ('font-style: italic;\n' if pd.isna(v) else '') +
            ('font-weight: bold' if s.name == "Ground Truth" else
            'color: red' if not eqna(v, example_predictions.loc["Ground Truth", k]) else
            'color: green' if not pd.isna(v) else '')
            for k, v in s.items()
        ],
        axis=1
    )
    display(example_predictions_styled)

In [86]:
display_predictions("train", 14)

Example address (train 14):
Ffm


,City,house,Name
libpostal,nan,ffm,nan
deepparse,ffm,nan,nan
xlm-roberta-large,nan,nan,Ffm
SpaCy,Ffm,nan,nan
Llama-3-8B-prompt0-0shot,Ffm,nan,nan
Mistral-3-8B-Instruct-prompt0-0shot,nan,nan,nan
Qwen3.5-9B-prompt0-0shot,nan,nan,nan
Llama-3-8B-model-best,Ffm,nan,nan
Mistral-3-8B-Instruct-model-best,Ffm,nan,nan
Qwen3.5-9B-best-from-optuna,Ffm,nan,nan


In [87]:
display_predictions("val", 112)

Example address (val 112):
3/51 Roscoe Street, Bondi Sydney/Austr.


,City,Country,HouseNumber,State,StreetName,Neighborhood
libpostal,nan,nan,3/51,nan,roscoe street bondi sydney/austr.,nan
deepparse,sydney/austr.,nan,3/51,nan,roscoe street bondi,nan
xlm-roberta-large,Bondi,Austr.,3/51,Sydney,Roscoe Street,nan
SpaCy,nan,nan,3/51,nan,"Roscoe Street, Bondi Sydney/Austr.",nan
Llama-3-8B-prompt0-0shot,Sydney,nan,3/51,Austr.,Roscoe Street,nan
Mistral-3-8B-Instruct-prompt0-0shot,Sydney,Australia,3/51,nan,Roscoe Street,nan
Qwen3.5-9B-prompt0-0shot,Sydney,Australia,3/51,nan,Roscoe Street,Bondi
Llama-3-8B-model-best,Bondi,Austr.,3/51,Sydney,Roscoe Street,nan
Mistral-3-8B-Instruct-model-best,Bondi Sydney,Austr.,3/51,nan,Roscoe Street,nan
Qwen3.5-9B-best-from-optuna,Sydney,Austr.,3/51,nan,Roscoe Street,Bondi


In [88]:
display_predictions("train", 462)

Example address (train 462):
Terbetscha, Lemberg,(Oesterreich) Polen


,City,Country,HouseNumber,StreetName,Name,OOV,Neighborhood,Other
libpostal,lemberg,oesterreich polen,nan,terbetscha,nan,nan,nan,nan
deepparse,nan,nan,nan,terbetscha lemberg(oesterreich) polen,nan,nan,nan,nan
xlm-roberta-large,Lemberg,Polen,nan,,Terbetscha,Oesterreich,nan,nan
SpaCy,Terbetscha,"Lemberg,(Oesterreich) Polen",nan,nan,nan,nan,nan,nan
Llama-3-8B-prompt0-0shot,Terbetscha,Polen,nan,nan,nan,nan,nan,nan
Mistral-3-8B-Instruct-prompt0-0shot,Lemberg,Polen,nan,nan,nan,nan,nan,nan
Qwen3.5-9B-prompt0-0shot,Terbetscha,Polen,nan,nan,nan,nan,nan,nan
Llama-3-8B-model-best,Lemberg,Polen,Terbetscha,nan,nan,nan,nan,nan
Mistral-3-8B-Instruct-model-best,"Terbetscha, Lemberg",Oesterreich,nan,nan,nan,nan,nan,nan
Qwen3.5-9B-best-from-optuna,Terbetscha,nan,nan,nan,nan,nan,nan,"Lemberg,(Oesterreich) Polen"


In [89]:
display_predictions("val", 54)


Example address (val 54):
Madiorban-Hedersch/Ung.


,City,Country,StreetName,house,Name
libpostal,nan,nan,nan,madiorban-hedersch/ung.,nan
deepparse,madiorban-hedersch/ung.,nan,nan,nan,nan
xlm-roberta-large,,nan,nan,nan,Madiorban-Hedersch/Ung
SpaCy,Madiorban-Hedersch,nan,nan,nan,nan
Llama-3-8B-prompt0-0shot,Madiorban-Hedersch,nan,Hedersch/Ung,nan,nan
Mistral-3-8B-Instruct-prompt0-0shot,nan,nan,nan,nan,nan
Qwen3.5-9B-prompt0-0shot,nan,nan,nan,nan,nan
Llama-3-8B-model-best,Madiorban-Hedersch,Ung.,nan,nan,nan
Mistral-3-8B-Instruct-model-best,Madiorban-Hedersch,Ung.,nan,nan,nan
Qwen3.5-9B-best-from-optuna,Madiorban-Hedersch,nan,nan,nan,nan


In [90]:
display_predictions("val", 143)

Example address (val 143):
Schikun Nowe-Nejman Doar Ramat aim 54 Israel


,City,Country,HouseNumber,StreetName,house,Name,District,Neighborhood
libpostal,nan,nan,54,israel,schikun nowe-nejman doar ramat aim,nan,nan,nan
deepparse,israel,nan,54,schikun nowe-nejman doar ramat aim,nan,nan,nan,nan
xlm-roberta-large,nan,Israel,54,Doar Ramat aim,nan,Schikun Nowe-Nejman,nan,nan
SpaCy,nan,Israel,54,Doar Ramat aim,nan,nan,Nowe-Nejman,Schikun
Llama-3-8B-prompt0-0shot,Doar,Israel,54,Ramat aim,nan,nan,nan,Nowe-Nejman
Mistral-3-8B-Instruct-prompt0-0shot,nan,Israel,54,Ramat aim,nan,nan,nan,Nowe-Nejman
Qwen3.5-9B-prompt0-0shot,Nowe-Nejman,Israel,54,Ramat aim,nan,nan,Schikun,Doar
Llama-3-8B-model-best,Ramat aim,Israel,54,Schikun Nowe-Nejman Doar,nan,nan,nan,nan
Mistral-3-8B-Instruct-model-best,nan,Israel,54,Schikun Nowe-Nejman,nan,nan,nan,nan
Qwen3.5-9B-best-from-optuna,nan,Israel,54,Schikun Nowe-Nejman,nan,nan,nan,Ramat aim


In [103]:
for idx, row in bzkopen_merged[bzkopen_merged['field'] == 'VictimDeathPlace'].iterrows():
    for l in ["HouseNumber", "StreetName", "Neighborhood", "City", "Country"]:
        if not eqna(row[l], predictions_dfs["Qwen3.5-9B-best-from-optuna"].loc[idx, l]):
            display_predictions(row.name[0], row.name[1])
            break

Example address (train 25):
Auschwitz umgekommen


,City,StreetName,house,Name
libpostal,nan,nan,auschwitz umgekommen,nan
deepparse,auschwitz umgekommen,nan,nan,nan
xlm-roberta-large,nan,nan,nan,Auschwitz umgekommen
SpaCy,Auschwitz,nan,nan,nan
Llama-3-8B-prompt0-0shot,"umgekommen"" (note: ""umgekommen",Auschwitz,nan,nan
Mistral-3-8B-Instruct-prompt0-0shot,nan,nan,nan,nan
Qwen3.5-9B-prompt0-0shot,nan,nan,nan,nan
Llama-3-8B-model-best,Auschwitz,nan,nan,nan
Mistral-3-8B-Instruct-model-best,nan,nan,nan,nan
Qwen3.5-9B-best-from-optuna,Auschwitz,nan,nan,nan


Example address (train 28):
Auschwitz


,City
libpostal,auschwitz
deepparse,auschwitz
xlm-roberta-large,Auschwitz
SpaCy,Auschwitz
Llama-3-8B-prompt0-0shot,Auschwitz
Mistral-3-8B-Instruct-prompt0-0shot,nan
Qwen3.5-9B-prompt0-0shot,Auschwitz
Llama-3-8B-model-best,nan
Mistral-3-8B-Instruct-model-best,nan
Qwen3.5-9B-best-from-optuna,Auschwitz


Example address (train 31):
Auschwitz


...

,City,Country,HouseNumber,StreetName,Name
libpostal,auschwitz/polen,k.z.,nan,nan,nan
deepparse,k.z. auschwitz/polen,nan,nan,nan,nan
xlm-roberta-large,Auschwitz,Polen,nan,nan,K.Z.
SpaCy,K.Z. Auschwitz,Polen,nan,nan,nan
Llama-3-8B-prompt0-0shot,Auschwitz,Polen,K.Z.,Auschwitz,nan
Mistral-3-8B-Instruct-prompt0-0shot,Auschwitz,Polen,nan,nan,nan
Qwen3.5-9B-prompt0-0shot,Auschwitz,Polen,nan,nan,nan
Llama-3-8B-model-best,nan,Polen,nan,nan,nan
Mistral-3-8B-Instruct-model-best,nan,nan,nan,nan,nan
Qwen3.5-9B-best-from-optuna,K.Z. Auschwitz,Polen,nan,nan,nan


Example address (train 686):
KZ. Mauthausen


,City,Country,HouseNumber,StreetName,Name,District
libpostal,mauthausen,kz.,nan,nan,nan,nan
deepparse,kz. mauthausen,nan,nan,nan,nan,nan
xlm-roberta-large,Mauthausen,nan,nan,nan,KZ.,nan
SpaCy,nan,nan,nan,nan,nan,Mauthausen
Llama-3-8B-prompt0-0shot,nan,nan,KZ,Mauthausen,nan,nan
Mistral-3-8B-Instruct-prompt0-0shot,nan,nan,nan,nan,nan,nan
Qwen3.5-9B-prompt0-0shot,nan,nan,nan,nan,nan,nan
Llama-3-8B-model-best,nan,nan,nan,nan,nan,nan
Mistral-3-8B-Instruct-model-best,nan,nan,nan,nan,nan,nan
Qwen3.5-9B-best-from-optuna,Mauthausen,nan,nan,nan,nan,nan


Example address (test 49):
KZ. Straßenhof b. Riga


,City,Country,HouseNumber,StreetName,Name,Neighborhood,Other
libpostal,straßenhof b. riga,kz.,nan,nan,nan,nan,nan
deepparse,riga,nan,nan,kz. straßenhof b.,nan,nan,nan
xlm-roberta-large,Riga,nan,nan,,KZ. Straßenhof b,nan,nan
SpaCy,KZ. Straßenhof,nan,nan,nan,nan,nan,nan
Llama-3-8B-prompt0-0shot,nan,Latvia,KZ,Straßenhof,nan,b. Riga,nan
Mistral-3-8B-Instruct-prompt0-0shot,nan,nan,nan,Straßenhof,nan,nan,nan
Qwen3.5-9B-prompt0-0shot,Riga,nan,KZ,Straßenhof,nan,nan,nan
Llama-3-8B-model-best,nan,RIGA,nan,nan,nan,nan,nan
Mistral-3-8B-Instruct-model-best,nan,nan,nan,nan,nan,nan,nan
Qwen3.5-9B-best-from-optuna,Straßenhof,nan,nan,nan,nan,nan,b. Riga
